# DCSS và ADA-IM-DCSS — kiểm chứng Fashion-MNIST đầy đủ

**Bản tiếp tục đa thiết bị — 10/11/2026.** Giữ nguyên giao thức đã chạy trên T4.

Chọn **Python 3 → GPU A100 → phiên bản mới nhất** như ảnh cấu hình của bạn;
RAM cao cấp có thể để tắt. Sau đó **Runtime → Run all**, cho phép gắn Drive.
Nếu A100 không được cấp, chọn L4/T4/G4 hoặc GPU CUDA khác đang có. Notebook
tự tải dữ liệu, kiểm tra tính toàn vẹn, chạy các seed và lưu kết quả. Không cần
clone repo; không cài lại PyTorch/CUDA. Có CPU fallback bằng **cùng mạng PyTorch**,
nhưng sẽ chậm hơn. Nếu phiên ngắt, chọn runtime mới rồi Run all để tiếp tục từ
checkpoint gần nhất. Các lượt đã hoàn tất không chạy lại.

**Tự tìm lại checkpoint T4:** giữ nguyên thư mục `DCSS_FashionMNIST_Full` trên
cùng tài khoản Drive. Mã tự nhận bản V1 tương thích và giữ nguyên thư mục/định
danh cũ. Checkpoint V1 được giữ lại; V2 thêm hai slot dự phòng có SHA-256.
Đổi GPU không đổi tên lượt chạy. Nếu có nhiều lượt tương thích, notebook liệt
kê để bạn chỉ định thư mục, không chọn theo điểm tốt nhất.

**Giới hạn:** notebook không tự xin GPU mới khi Colab thu hồi phiên và không
vượt hạn mức thuê bao. Bạn cần chọn GPU còn được cấp rồi Run all. Chuyển thiết
bị/phần mềm không bảo đảm kết quả trùng từng bit; mọi phiên được ghi lại.
TPU không thuộc bộ thực thi PyTorch CUDA/CPU này. Chỉ mở một phiên ghi cùng
thư mục kết quả tại một thời điểm.

Mặc định: `paper70k` (10 seed, 100+100 epoch, 4 phương pháp), sau đó
`legacy10k` và `legacy70k` (mỗi bộ 3 seed, 80+40 epoch, 3 phương pháp).
Toàn chiến dịch gồm **58 lượt phân cụm**, có thể cần nhiều phiên Colab.

**Điểm cần hiểu trước khi chạy:** mã tác giả công bố và phương trình DCSS có
bất nhất. Bộ `paper70k` là tái triển khai theo phương trình với lựa chọn được
ghi rõ bên dưới, chưa thể gọi là tái lập nguyên trạng bảng kết quả bài báo.
70.000 ảnh gồm train và test chính thức: đây là phân cụm chuyển dẫn, không phải
đánh giá trên ảnh chưa từng dùng. Các chỉ số sẽ chỉ xuất hiện sau khi chạy thật.


# Giao thức kiểm chứng DCSS và ADA-IM-DCSS trên Fashion-MNIST

Giao thức khoa học chốt ngày 09/11/2026; cập nhật cách vận hành ngày 10/11/2026
để tiếp tục checkpoint T4 trên A100, GPU CUDA khác hoặc CPU. Bản cập nhật không
thay đổi dữ liệu, cấu trúc mạng, batch, hàm mục tiêu, số epoch hay seed. Tệp này
mô tả những gì chương trình thực hiện và giới hạn của kết luận được phép rút ra.

## 1. Câu hỏi và phạm vi kết luận

Mục tiêu thứ nhất là đối chiếu DCSS trên đủ 70.000 ảnh với bài báo.
Mục tiêu thứ hai là kiểm tra ADA-IM-DCSS có duy trì lợi ích khi tăng quy mô dữ
liệu, đồng thời xác định thành phần ADA có đem lại lợi ích so với chỉ dùng IM.
Hai mục tiêu cần các phép so sánh riêng, vì thay đổi đồng thời kiến trúc, số
epoch và số ảnh sẽ không cho biết yếu tố nào gây ra thay đổi kết quả.

Notebook mặc định chạy ba hồ sơ cấu hình, theo thứ tự:

| Hồ sơ | Ảnh | Kiến trúc | Tiền huấn luyện / phân cụm | Seed | Phương pháp |
|---|---:|---|---:|---|---|
| `paper70k` | 70.000 | Conv/BN theo mã tác giả, khởi tạo He theo bài báo | 100 / 100 | 0–9 | DCSS theo phương trình; DCSS-RMS; IM-DCSS; ADA-IM-DCSS |
| `legacy10k` | 10.000 ảnh test chính thức | Conv không BN, đầu ra sigmoid theo bản cục bộ | 80 / 40 | 0, 1, 2 | DCSS-RMS; IM-DCSS; ADA-IM-DCSS |
| `legacy70k` | 70.000 | Giống `legacy10k` | 80 / 40 | 0, 1, 2 | Giống `legacy10k` |

Tổng cộng 58 lượt phân cụm và 16 lượt tiền huấn luyện dùng chung. Đây là một
chiến dịch nhiều lượt, có thể cần nhiều phiên Colab. Nhật ký cũ cho thấy đã
khởi chạy trên T4, nhưng chưa có bảng thời lượng toàn chiến dịch được xác minh.
Chương trình chỉ ước lượng thời gian còn lại của pha hiện tại sau khi
đã đo epoch. Không hạ số epoch/seed âm thầm để hoàn thành trong một phiên.

`legacy10k` được chạy mới bằng cùng bộ thực thi với `legacy70k`. Nó kế thừa
kiến trúc, siêu tham số và hàm mục tiêu cũ, nhưng lịch trộn mẫu, việc bỏ nhánh
residual không dùng và trạng thái ngẫu nhiên khác bộ thực thi cũ. Vì vậy không
đòi hỏi tái tạo đúng từng chữ số của bảng kết quả cũ. Hai lượt chạy mới mới là
cặp phù hợp để đánh giá tác động của tăng quy mô dữ liệu.

Tập 70.000 được nối theo thứ tự **train 60.000 → test 10.000**. Huấn luyện
không dùng nhãn; nhãn chỉ kiểm tra dữ liệu và tính chỉ số sau huấn luyện.
Đây là phân cụm chuyển dẫn (*transductive clustering*), không phải thử nghiệm
khái quát hóa trên tập kiểm tra độc lập. Hơn nữa, 10.000 ảnh đã dùng trước đây
để phát triển phương pháp nằm trong bộ 70.000; không được gọi bộ 70.000 là tập
chưa từng được sử dụng trong quá trình nghiên cứu.

## 2. Đối chiếu bài báo, mã tác giả và lựa chọn triển khai

Nguồn chính:

- [DCSS, Machine Learning 115:81, 2026](https://link.springer.com/article/10.1007/s10994-026-07026-w), phương trình (13)–(22), mục 5.3 và 5.5, bảng 2.
- [Mã tác giả tại commit cố định](https://github.com/gvardakas/Soft-Silhouette/tree/fa3d84c0f3e7c682729cd5ba6f856bb3ef532637).
- [RIM: Discriminative Clustering by Regularized Information Maximization, NeurIPS 2010](https://proceedings.neurips.cc/paper/2010/hash/42998cf32d552343bc8e460416382dca-Abstract.html).
- [SHOT, ICML 2020, phương trình (3)](https://proceedings.mlr.press/v119/liang20a/liang20a.pdf).
- [Fashion-MNIST chính thức](https://github.com/zalandoresearch/fashion-mnist).

Đường dẫn RIM được cung cấp ban đầu có mã băm khác; liên kết phía trên là trang
của đúng bài của Krause, Perona và Gomes. RIM cung cấp cơ sở tối đa hóa thông
tin tương hỗ cho phân cụm; SHOT sử dụng entropy có điều kiện và entropy biên
trong thích nghi miền. Không xem RIM/SHOT là chứng minh cho các hệ số, lịch
warmup hay quy tắc băng thông ADA cụ thể của đề xuất này.

| Nội dung | Bài báo / mã công bố | Quyết định trong `paper70k` |
|---|---|---|
| Soft silhouette | (13) chia khoảng cách có trọng số cho tổng xác suất, loại mẫu đang xét khỏi mẫu số. `Objectives.py` công bố thiếu phép chia này và có cách xếp chỉ số riêng. | Theo (13)–(17), có kiểm tra bằng phép tính vòng lặp độc lập và kiểm tra gradient. |
| Softmax RBF | Bài báo: nhân hệ số T=20. `forward_softMax` công bố mặc định chia 0,25, tương đương T=4. | Chia 0,05, tương đương T=20. |
| Băng thông | Bài báo: giá trị dương nhỏ, tham số được học; không chỉ rõ trị số. Mã công bố khởi tạo log_sigma=0. | `dcss_paper`: sigma=1 rồi học log_sigma. Không gọi đây là sigma cố định. |
| Kiến trúc ảnh | Mã tác giả có BatchNorm sau các khối và sau tanh, kể cả đầu ra giải mã. Bản cục bộ không có BN, đầu ra sigmoid. | Dùng cấu trúc Conv/BN của mã tác giả cho cả bốn phương pháp; không trộn hai kiến trúc trong cùng hồ sơ. |
| Khởi tạo mạng | Bài báo nêu He; mã lớp công bố không có lời gọi He rõ ràng. | He cho Conv/Linear; lựa chọn được ghi trong manifest. |
| MSE | Công thức mô tả tổng sai số trên chiều; mã tác giả dùng `nn.MSELoss` lấy trung bình cả pixel. | Trung bình pixel theo mã, tránh thay tỷ lệ 784 lần mà không công bố. |
| Entropy | Bài báo không nêu cơ số log một cách tường minh; mã dùng cơ số 2. | Log tự nhiên, trung bình mẫu, theo công thức thường dùng cho IM và bản cục bộ. Chưa coi tương đương hệ số với entropy cơ số 2. |
| Tiền xử lý | Không có bộ nạp Fashion-MNIST đầy đủ trong các tệp tác giả đã kiểm tra; loader EMNIST có MinMaxScaler. | Pixel / 255, không tăng cường ảnh, không chuẩn hóa từng ảnh. Đây là lựa chọn triển khai được công bố. |
| Đánh giá và BN | Mã công bố thu dự đoán trong quá trình cập nhật theo batch. | Một lượt `eval()` sau epoch cuối; không cập nhật BN/ADA khi đánh giá. |

Vì các bất nhất trên, tên đúng của kết quả là **tái triển khai DCSS theo phương
trình với các lựa chọn được công bố**, chưa phải tái lập nguyên trạng chương
trình đã sinh bảng 2. Nếu kết quả lệch bảng 2, cần đối chiếu những lựa chọn này
với tác giả trước khi kết luận phương pháp gốc thất bại. Không hứa đạt lại
điểm công bố. Bảng 2 báo cáo Fashion-MNIST NMI 0,63 ± 0,02 và ARI 0,45 ± 0,03;
đây chỉ là mốc tham khảo, không được đưa vào cột kết quả thực nghiệm mới.

Có thêm `author_objective_control` trong mã để chẩn đoán khác biệt: giữ biểu
thức/indexing soft silhouette của mã công bố, entropy cơ số 2 và temperature
0,25. Đây vẫn là phép kiểm soát hàm mục tiêu trên bộ thực thi chung, **không**
được ghi là chạy nguyên xi toàn bộ repository tác giả. Biến thể này tắt mặc
định; nếu bật phải tạo hồ sơ mới, không thay thế kết quả chính sau khi xem điểm.

## 3. Hàm mục tiêu và nội dung đề xuất

Với batch B ảnh, K=10, q là xác suất phân cụm:

```text
a_ik = sum_{j != i} q_jk ||z_i-z_j|| / max(sum_{j != i} q_jk, epsilon)
b_ik = min_{l != k} a_il
S    = mean_i sum_k q_ik (b_ik-a_ik) / max(a_ik,b_ik,epsilon)
q_ik = softmax_k(exp(-||z_i-c_k||² / sigma_k²) / 0.05)
Hc   = -mean_i sum_k q_ik ln(q_ik)
qbar = mean_i q_i
KL   = sum_k qbar_k ln(K*qbar_k)
```

DCSS: `L = MSE + 0.01*(1-S) + 0.01*Hc`.

IM-DCSS và ADA-IM-DCSS:
`L = MSE + 0.012*(1-S) + 0.0025*omega(t)*Hc + 0.004*KL`.
`omega=0` trong 70% đầu số epoch phân cụm, sau đó tăng theo nửa đường cosine
đến 1 ở epoch cuối. KL nhỏ khuyến khích phân bố cụm cân bằng; Hc nhỏ khuyến
khích gán cụm dứt khoát. Với hệ số bằng nhau và không warmup, tổng hai thành
phần entropy tương đương âm thông tin tương hỗ cộng hằng số. Ở cấu hình hiện
tại, cần gọi là **điều chuẩn IM có trọng số và lịch điều chỉnh**, không gọi là
đúng nguyên dạng hàm âm thông tin tương hỗ của RIM/SHOT.

ADA giữ cơ chế đang có: tâm cụm học bằng gradient; sigma khởi tạo theo phân
tán từng cụm, cập nhật bằng EMA hệ số 0,15; mục tiêu sigma kết hợp RMS toàn cục,
RMS nội cụm mềm và hiệu chỉnh khối lượng cụm nhẹ. Ngưỡng bảo vệ trung bình RBF
là 0,08 / 0,78. Sau epoch 1 và 3, chỉ nếu tiêu chí không nhãn phát hiện suy biến
thì đặt lại sigma bằng RMS. Mọi lần bảo vệ/phục hồi đều được đếm và ghi lại.
EMA dùng thống kê trước bước tối ưu nhưng thực hiện cập nhật buffer sau
`backward/step`, để tránh sửa trạng thái đang được autograd sử dụng.

`normalize_latent_scale` trong cấu hình cũ chỉ được lưu mà không có phép chuẩn
hóa thực sự trong forward. Bản này không tuyên bố đã thêm chuẩn hóa latent.
Các ngưỡng, warmup và hệ số ADA được giữ từ bản cũ; chưa có chứng cứ rằng đó là
cấu hình tối ưu hoặc là đóng góp mới đã được xác nhận bởi tổng quan tài liệu.

DCSS-RMS là đối chứng cần thiết: nó cũng khởi tạo sigma theo RMS rồi học bằng
gradient, giới hạn [0,15; 4,5], nhưng không có EMA/guard/rescue. Nhờ đối chứng
này, việc thắng DCSS sigma khởi tạo 1 không bị hiểu nhầm thành bằng chứng toàn
bộ lợi ích đến từ ADA. Đặc biệt, ADA-IM so với IM-DCSS giữ nguyên các hệ số hàm
mục tiêu; cặp này kiểm tra đóng góp của **cả gói ADA** gồm khởi tạo, EMA, guard
và rescue. Chưa cô lập riêng từng thành phần của gói ADA. So DCSS với IM còn
thay đổi hệ số silhouette/entropy, nên chưa phải phép loại bỏ thành phần IM
thuần túy. `ada_only` có sẵn như một đối chứng bổ sung, tắt mặc định.

## 4. Kiểm soát thực nghiệm và tài nguyên

Mỗi seed tiền huấn luyện đúng một lần, lấy K-means với `n_init=100`, rồi chia
cùng trọng số và tâm khởi tạo cho các phương pháp của seed đó. Thứ tự batch
cũng giống nhau giữa các phương pháp. Adam dùng beta=(0,9; 0,999), LR tiền
huấn luyện 1e-3 và weight decay 1e-5; LR phân cụm 5e-4, không weight decay.
Batch 256, latent 10. Hồ sơ paper không cắt gradient; hồ sơ legacy cắt norm
5 ở pha phân cụm theo bản cũ. Không early stopping, không chọn seed tốt nhất,
không chọn checkpoint theo nhãn thật, không điều chỉnh hệ số từ kết quả 70k.

Ảnh lưu uint8 trên RAM máy; chỉ batch hiện tại được đổi float32 và chuyển GPU.
Soft silhouette tính trong batch, không tạo ma trận 70.000 × 70.000. Khoảng
cách N × K chỉ dùng khi khởi tạo/chẩn đoán tâm; đánh giá encoder cũng theo
batch. FP32, AMP/TF32 tắt; bật thuật toán xác định. Nếu CUDA báo thao tác không
hỗ trợ chế độ xác định, chương trình dừng thay vì tự đổi chế độ. Mỗi epoch lưu
thời gian, mức sử dụng VRAM đỉnh của PyTorch và chẩn đoán không nhãn.

Giữ bản torch/CUDA đang cài trên Colab. Không ép hạ cấp hoặc cài lại torch,
torchvision. Lưu phiên bản Python, torch, CUDA, cuDNN, GPU và thư viện tính
toán thực tế. A100 là lựa chọn ưu tiên trong giao diện, không phải yêu cầu cứng
trong mã: dùng GPU CUDA mà Colab thực sự cấp. Không tăng batch hoặc bật AMP/TF32
chỉ vì A100 mạnh hơn, vì việc đó sẽ đổi thí nghiệm đang tiếp tục. RAM cao cấp
không bắt buộc với cách nạp ảnh uint8/batch hiện tại.

`DEVICE="auto"` ưu tiên CUDA, nếu không có thì `ALLOW_CPU_FALLBACK=True` cho
chạy cùng mạng PyTorch trên CPU. Đây không phải mô hình NumPy thay thế. CPU có
thể rất chậm; đặt `ALLOW_CPU_FALLBACK=False` nếu chỉ muốn chạy khi có GPU.
Không gọi một lượt có đoạn CPU là lượt chạy thuần A100. TPU cần bộ thực thi
XLA riêng và chưa được hỗ trợ. Khi Colab thu hồi runtime, notebook không thể
tự xin một GPU khác hoặc vượt hạn mức thuê bao; người dùng chọn thiết bị mới
rồi Run all. Thông tin giới hạn và GPU phụ thuộc tài nguyên của Colab:
[FAQ chính thức](https://research.google.com/colaboratory/faq.html#resource-limits).

## 5. Đánh giá đã định trước

Chỉ số chính: **ARI**. Giả thuyết chính của phần cải tiến là chênh lệch ARI
`ADA-IM − IM-DCSS` trên 10 seed của `paper70k`. ACC (ghép nhãn bằng Hungarian
chính xác) và NMI (trung bình số học) là chỉ số phụ. So với DCSS-paper và
DCSS-RMS là đối chiếu bổ sung; không dùng nhiều phép so sánh rồi chỉ báo cáo
phép có lợi. Nếu muốn kết luận xác nhận cho nhiều giả thuyết thì cần hiệu
chỉnh đa kiểm định trong một giao thức bổ sung.

Bản vận hành V2 cho tiếp tục qua nhiều loại GPU/phần mềm để không bỏ công đã
chạy. Vì thế, ngoài chỉ số chất lượng phải kiểm tra `execution_trace`,
`mixed_environment` và `campaign_mixed_environment`. Cột `planned_primary`
giữ giả thuyết chính đã chốt; cột `confirmatory` chỉ bật khi đủ cặp seed và
không phát hiện pha trộn môi trường. Kết quả có đổi môi trường vẫn giữ đầy đủ,
nhưng phép suy luận mang tính thăm dò cho đến khi đánh giá được ảnh hưởng của
thay đổi đó. Không so trực tiếp thời gian tổng của lượt T4/A100/CPU pha trộn để
khẳng định thuật toán nhanh hơn. Nếu muốn kết luận tốc độ hoặc cải tiến nhỏ,
nên bổ sung các cặp chạy trong cùng điều kiện, báo cáo riêng; không xóa lượt cũ.

Báo cáo mọi seed gồm cả suy biến: trung bình, độ lệch chuẩn mẫu, chênh lệch
theo cặp seed, khoảng tin cậy t 95%, kiểm định đổi dấu chính xác hai phía.
Khoảng tin cậy mô tả biến thiên do seed trên tập dữ liệu cố định, không phải
độ bất định khi lấy mẫu một quần thể ảnh mới. Với 3 seed legacy, chỉ số thống
kê mang tính thăm dò; không đủ để tuyên bố cải tiến vững chắc.

Không kết luận ADA có lợi chỉ vì trung bình cao hơn một chút. Cần xem khoảng
tin cậy chênh lệch chính, tỷ lệ suy biến, từng seed, thời gian và VRAM. Nếu ADA
không hơn IM, cần báo cáo đúng và xem lại độ cần thiết của thành phần ADA.
Nếu chưa đủ seed, báo cáo là **chưa hoàn tất**. Không loại lượt chạy kém, không
ghi lượt lỗi thành điểm 0 hoặc bỏ qua lỗi mà vẫn coi chiến dịch thành công.

## 6. Dữ liệu và bằng chứng đầu ra

Bốn tệp Drive được so byte bằng SHA-256 với bản giải nén từ kho Fashion-MNIST
chính thức; các bản nén chính thức được kiểm tra MD5 theo bảng phát hành.
`audit/dataset_verification.json` ghi kết quả so khớp, magic/header/chiều dài
IDX và số lượng lớp. Mỗi lần chạy kiểm tra lại SHA-256, số ảnh và thứ tự nối.

Mỗi thư mục kết quả có `manifest.json`, bản mã `experiment.py`, `status.json`,
`completeness.json`, `per_seed.csv`, `summary.csv`, `paired_comparisons.json`.
Trong từng seed có trọng số tiền huấn luyện, tâm K-means dùng chung; mỗi phương
pháp có checkpoint, `history.json`, `result.json`, `predictions.npz` gồm dự
đoán, xác suất, embedding và nhãn phục vụ kiểm tra độc lập. Đơn vị chỉ số trong
CSV/JSON là tỷ lệ 0–1, khi vẽ mới nhân 100. ARI có thể âm.

Checkpoint V2 lưu model, buffer BN/ADA, optimizer, RNG, epoch/batch tiếp theo,
tổng loss và số mẫu của phần epoch đã chạy, thời gian tích lũy và lịch sử thiết
bị. Lưu ở cuối mỗi epoch và sau một bước tối ưu khi quá 300 giây kể từ lần
lưu trước. Phiên kết thúc đột ngột sẽ chạy lại phần sau checkpoint gần nhất;
không bảo đảm lưu được bước đang thực hiện ngay lúc Colab thu hồi máy.

Hai slot `checkpoint_A.pt` và `checkpoint_B.pt` ghi luân phiên bằng tệp tạm.
`checkpoint_index.json` trỏ tới bản đã ghi xong kèm SHA-256, giữ tham chiếu bản
trước. Nếu bản mới hỏng, thử phục hồi bản trước và thông báo. Nếu cả hai không
đọc được, chương trình dừng để xử lý, không tự bắt đầu lại. Cơ chế này giảm
rủi ro khi ghi gián đoạn, không thay thế sao lưu hoặc bảo đảm tuyệt đối cho
hệ thống tệp Drive. Chỉ chạy một phiên ghi cùng thư mục tại một thời điểm.

Định danh V2 gắn với cấu hình khoa học, dữ liệu, các hệ số phương pháp và mã
AST của các phép tính cốt lõi; GPU/CPU, hệ điều hành và phiên bản thư viện nằm
trong nhật ký riêng. Đổi thiết bị không tạo lại thí nghiệm. Đổi batch, kiến
trúc, loss, số epoch hoặc dữ liệu không được phép tiếp tục chung. Nếu tìm
thấy tiến độ khác cấu hình, chương trình yêu cầu xử lý rõ ràng thay vì âm thầm
tạo lượt chạy mới. Nếu có nhiều thư mục tương thích, chỉ định `RESUME_FOLDERS`;
không lựa chọn theo điểm hoặc ghép seed từ các thư mục khác nhau.

Đối với dữ liệu đã chạy bằng V1, chỉ tự nhận bản mã có SHA-256
`b29c6c15ccc41ac03adea4edca8527f9971c70ab923c291d81c889421bbe5703`, cấu hình và
dữ liệu khớp hoàn toàn, mã toán học khớp V2. Mã tính lại định danh cũ từ
manifest, giữ nguyên thư mục, `manifest.json`, `experiment.py`, checkpoint V1
và kết quả đã xong; thêm `portable_resume.json` để ghi ánh xạ tương thích.
Các checkpoint mới dùng hai slot V2. Notebook liệt kê tiến độ trước huấn luyện
để người dùng phát hiện việc gắn nhầm tài khoản Drive. Công cụ kết nối trong
phiên chuẩn bị chưa tìm thấy thư mục kết quả qua tìm kiếm Drive; vì vậy không
tuyên bố đã xác minh trực tiếp các checkpoint T4 thật của người dùng. Phép
kiểm chứng nhập V1 được làm trên checkpoint tạo bằng mã V1 đã lưu nguyên bản.

Trạng thái được đọc về CPU trước, sau đó đưa model và các moment của Adam lên
thiết bị đích; bộ đếm Adam không capturable giữ trên CPU. Đây là cách tiếp tục
cả model lẫn optimizer, không phải chỉ tải trọng số rồi tạo optimizer mới.
Xem [hướng dẫn checkpoint và chuyển thiết bị của PyTorch](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).

Mỗi lần tiếp tục tạo một mục `sessions/`, bản mã trong `implementations/` và
dấu mốc ở từng pha. `ALLOW_SOFTWARE_CHANGE=True` cho phép thử tiếp tục khi
Colab đổi phiên bản, với nhật ký thay đổi; nếu định dạng hoặc toán tử không
tương thích vẫn dừng có thông báo. Đặt False để yêu cầu khớp phần mềm. Khôi
phục RNG CUDA trên thiết bị 0 nếu hợp lệ; checkpoint CPU không có RNG CUDA
thì khởi tạo lại RNG GPU theo seed và ghi rõ. Mạng hiện tại không có dropout,
thứ tự batch được tạo độc lập trên CPU theo seed/epoch.

Tính tiếp tục của trạng thái không đồng nghĩa với tái tạo từng bit. PyTorch
không bảo đảm tái lập tuyệt đối giữa thiết bị, nền tảng hoặc phiên bản:
[tài liệu về khả năng tái lập](https://docs.pytorch.org/docs/2.15/notes/randomness.html).
Notebook thực hiện kiểm tra ngắn chuyển optimizer thiết bị hiện tại → CPU →
thiết bị hiện tại trước khi chạy dài, đồng thời thử batch 256 trên GPU. Đây là
kiểm tra kỹ thuật của đúng runtime được cấp, không phải số liệu chất lượng.

`MAX_SESSION_HOURS=0` không giới hạn thêm ngoài Colab. Giá trị dương cho phép
tạm dừng tại ranh giới bước tối ưu, lưu checkpoint và xuất kết quả tạm. Dừng
bằng nút Stop hoặc mất phiên sử dụng checkpoint đã ghi trước đó. NaN/hết bộ
nhớ không được xử lý bằng cách tự giảm batch. Phiên mới chọn GPU khác hoặc
CPU rồi Run all; các lượt đã hoàn tất bỏ qua cả bước dựng/pretrain mạng.

## 7. Những việc còn cần trước khi đưa vào luận văn

Chưa xác minh được kết quả GPU đầy đủ ở lần cập nhật này. Đã quan sát lỗi
PyTorch CPU của notebook cũ; chưa chạy A100 trực tiếp trong phiên chuẩn bị.
Cần chạy
đủ các hồ sơ, kiểm tra `complete=true`, rồi phân tích chênh lệch. Bất nhất giữa
bài báo và mã tác giả cần được trình bày trung thực trong phần tái lập; có thể
xin tác giả xác nhận tiền xử lý, sigma, cơ số entropy, MSE, BN và mã đã sinh
bảng kết quả. Sau đó mới quyết định có cần một lượt tái lập bổ sung.

Kết quả trên một bộ dữ liệu chưa đủ khẳng định phương pháp tổng quát hay tính
mới. Bài toán mất cân bằng lớp, các bộ ảnh khác và phân tích riêng từng thành
phần ADA là những kiểm chứng tiếp theo, không được coi là đã hoàn thành chỉ
bằng Fashion-MNIST cân bằng.


## 1. Nhận diện A100/GPU khác hoặc CPU
Giữ PyTorch/CUDA được Colab cài sẵn. Không giới hạn tên GPU. Thiếu GPU được báo rõ, không gây lỗi bắt buộc T4 như V1.

In [ ]:
import os, sys, subprocess, importlib.util
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
import torch
print("PyTorch:", torch.__version__, "CUDA:", torch.version.cuda)
if torch.cuda.is_available():
    print("GPU được Colab cấp:", torch.cuda.get_device_name(0))
    print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))
else:
    if os.environ.get("COLAB_TPU_ADDR") or os.environ.get("TPU_NAME"):
        raise RuntimeError("Đang chọn TPU. Hãy đổi sang GPU A100/L4/T4 hoặc CPU; notebook này không dùng XLA.")
    print("Đang ở CPU. Có thể tiếp tục checkpoint bằng CPU; sẽ chậm. Muốn dùng A100, đổi loại runtime trước khi Run all.")
required = {"numpy": "numpy", "scipy": "scipy", "sklearn": "scikit-learn", "pandas": "pandas", "matplotlib": "matplotlib", "gdown": "gdown", "threadpoolctl": "threadpoolctl"}
missing = [package for module, package in required.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", *missing])
print("Dependencies ready. PyTorch was not replaced.")

## 2. Gắn đúng Drive và chính sách tiếp tục
Đổi thiết bị không đổi cấu hình thí nghiệm. Giữ nguyên OUTPUT_ROOT để tìm checkpoint T4 cũ. Kiểm tra dung lượng Drive; checkpoint còn lưu được mới có thể khôi phục.

In [ ]:
from google.colab import drive
from pathlib import Path
import json, hashlib, zipfile
drive.mount("/content/drive")
OUTPUT_ROOT = Path("/content/drive/MyDrive/DCSS_FashionMNIST_Full")
DATA_ROOT = Path("/content/fashion_mnist_verified")
RUN_PROFILES = ["paper70k", "legacy10k", "legacy70k"]
DEVICE = "auto"               # GPU CUDA nếu có, nếu không thì CPU.
ALLOW_CPU_FALLBACK = True      # Đặt False nếu chỉ muốn chờ GPU.
ALLOW_SOFTWARE_CHANGE = True   # Cho tiếp tục khi Colab đổi phiên bản; luôn ghi nhật ký.
CHECKPOINT_SECONDS = 300       # Ngoài cuối mỗi epoch, lưu khi qua 5 phút sau một bước tối ưu.
MAX_SESSION_HOURS = 0          # 0 = không đặt giới hạn riêng. Ví dụ 4 = tự lưu rồi tạm dừng sau ~4h.
RESUME_FOLDERS = {}            # Ví dụ {"paper70k": "paper70k_2cd93c725947"} nếu có nhiều lượt tương thích.
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
# Kiểm tra quyền ghi trước khi chạy dài.
probe = OUTPUT_ROOT / ".write_probe"
probe.write_text("ok", encoding="utf-8")
assert probe.read_text(encoding="utf-8") == "ok"
probe.unlink()
print("Kết quả sẽ được lưu tại:", OUTPUT_ROOT)
print("Các hồ sơ sẽ chạy:", RUN_PROFILES)

## 3. Mã thực thi được chốt cùng notebook
Ô dưới ghi module vào /content. Có thể mở rộng ô để kiểm tra. Module không dùng nhãn trong hàm huấn luyện.

In [ ]:
%%writefile /content/dcss_full_experiment.py
"""Auditable DCSS / ADA-IM experiments. Training functions never receive labels.

See PROTOCOL.md before interpreting a result. No claim of bitwise reproduction of
the published DCSS scores: the paper, released code and legacy code disagree.
"""
from __future__ import annotations

import copy
import ast
import gzip
import hashlib
import importlib.metadata
import itertools
import json
import math
import os
import platform
import random
import struct
import time
import uuid
import warnings
import urllib.request
from datetime import datetime, timezone
from dataclasses import asdict, dataclass, field, replace
from pathlib import Path

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
import numpy as np
import torch
from torch import nn
from scipy.optimize import linear_sum_assignment
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score, normalized_mutual_info_score
from sklearn.metrics import silhouette_score
from threadpoolctl import threadpool_limits

VERSION = "full-fmnist-2026-11-10-portable-v2"
LEGACY_CODE_SHA256 = "b29c6c15ccc41ac03adea4edca8527f9971c70ab923c291d81c889421bbe5703"
SCIENTIFIC_VERSION = "dcss-equations-ada-2026-11-09"
UPSTREAM_COMMIT = "fa3d84c0f3e7c682729cd5ba6f856bb3ef532637"
FILES = {
    "train-images-idx3-ubyte": ("1XjvEO46beyAhBPorMEaRaFozU1m6raSS", "8d4fb7e6c68d591d4c3dfef9ec88bf0d"),
    "train-labels-idx1-ubyte": ("1vMq9NhjOpFW71TxQB-VpAdc5UtykGLvL", "25c81989df183df01b3e8a0aad5dffbe"),
    "t10k-images-idx3-ubyte": ("12klOdWX0FAUq3WB7RkakYRZviZTG1BoI", "bef4ecab320f06d8554ea6380940ec79"),
    "t10k-labels-idx1-ubyte": ("1yXJ_8hlNKFrVPWclDQuInAvauEErtKGF", "bb300cfdad3c16e7a12a480ee83cd310"),
}


def digest(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()


def atomic_json(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    tmp.write_text(json.dumps(obj, indent=2, ensure_ascii=False, allow_nan=False), encoding="utf-8")
    os.replace(tmp, path)


def atomic_torch(path, obj):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_name(path.name + ".tmp")
    torch.save(obj, tmp)
    os.replace(tmp, path)


def prepare_data(folder, raw_hashes):
    """Prefer the supplied Drive files; official fallback is explicit and checked.

    raw_hashes are independently checked against the official gz files locally.
    Every Colab download must match those hashes. Never silently select 10k.
    """
    import gdown
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    sources = {}
    for name, (file_id, md5) in FILES.items():
        path = folder / name
        if not path.exists():
            tmp = path.with_name(name + ".download")
            try:
                result = gdown.download(id=file_id, output=str(tmp), quiet=False)
                if result is None:
                    raise RuntimeError("Drive did not return a file")
                sources[name] = "user_drive:" + file_id
            except Exception as exc:
                print(f"Drive download unavailable for {name}: {type(exc).__name__}; using official identical bytes.", flush=True)
                url = "https://raw.githubusercontent.com/zalandoresearch/fashion-mnist/master/data/fashion/" + name + ".gz"
                with urllib.request.urlopen(url, timeout=120) as r:
                    compressed = r.read()
                if hashlib.md5(compressed).hexdigest() != md5:
                    raise ValueError("Official compressed checksum mismatch: " + name)
                tmp.write_bytes(gzip.decompress(compressed))
                sources[name] = url
            if digest(tmp) != raw_hashes[name]:
                raise ValueError("Downloaded data differ from canonical Fashion-MNIST: " + name)
            os.replace(tmp, path)
        else:
            sources[name] = "verified_cache"
        if digest(path) != raw_hashes[name]:
            raise ValueError("Cached data checksum mismatch: " + name)
    atomic_json(folder / "download_sources.json", sources)
    return folder


def read_idx(path):
    raw = Path(path).read_bytes()
    if len(raw) < 8:
        raise ValueError("Truncated IDX")
    magic, n = struct.unpack(">II", raw[:8])
    if magic == 2051:
        if len(raw) < 16:
            raise ValueError("Truncated image header")
        rows, cols = struct.unpack(">II", raw[8:16])
        if (rows, cols) != (28, 28) or len(raw) != 16 + n * rows * cols:
            raise ValueError("Wrong IDX image shape/length")
        return np.frombuffer(raw, np.uint8, offset=16).reshape(n, 1, 28, 28).copy()
    if magic == 2049 and len(raw) == 8 + n:
        labels = np.frombuffer(raw, np.uint8, offset=8).copy()
        if np.any(labels > 9):
            raise ValueError("Label outside Fashion-MNIST 0..9")
        return labels
    raise ValueError("Wrong IDX magic/length")


def load_data(folder, expected_hashes, split="all70k"):
    folder = Path(folder)
    hashes = {n: digest(folder / n) for n in FILES}
    if hashes != expected_hashes:
        raise ValueError("Dataset hashes do not match independently verified Fashion-MNIST")
    a, b = read_idx(folder / "train-images-idx3-ubyte"), read_idx(folder / "train-labels-idx1-ubyte")
    c, d = read_idx(folder / "t10k-images-idx3-ubyte"), read_idx(folder / "t10k-labels-idx1-ubyte")
    if (len(a), len(b), len(c), len(d)) != (60000, 60000, 10000, 10000):
        raise ValueError("Expected exactly 60,000 training and 10,000 test images/labels")
    for y, n in [(b, 6000), (d, 1000)]:
        if not np.array_equal(np.bincount(y, minlength=10), np.full(10, n)):
            raise ValueError("Unexpected class counts")
    if split == "all70k":
        images, labels = np.concatenate([a, c]), np.concatenate([b, d])
    elif split == "train60k":
        images, labels = a, b
    elif split == "legacy10k":
        images, labels = c, d
    else:
        raise ValueError(split)
    # Keep uint8 in CPU RAM. Convert only a mini-batch to float32 on the device.
    meta = {"sha256": hashes, "split": split, "n": len(images), "order": "train_then_test" if split == "all70k" else split,
            "normalization": "float32(pixel)/255", "class_counts_audit_only": np.bincount(labels, minlength=10).tolist()}
    return torch.from_numpy(images), labels, meta


@dataclass(frozen=True)
class Method:
    name: str
    sigma: str = "one_trainable"
    sil: float = .01
    ent: float = .01
    balance: float = 0.
    warmup: float = 0.
    ema: float = .15
    rescue: bool = False
    objective: str = "paper_normalized"
    entropy_base: str = "e"
    temperature: float = .05


METHODS = {
    "dcss_paper": Method("dcss_paper"),
    "dcss_rms": Method("dcss_rms", sigma="rms_trainable"),
    "im_rms": Method("im_rms", sigma="rms_trainable", sil=.012, ent=.0025, balance=.004, warmup=.70),
    "ada_im": Method("ada_im", sigma="adaptive", sil=.012, ent=.0025, balance=.004, warmup=.70, rescue=True),
    # Optional controls: not run by default; not needed to select the best seed.
    "ada_only": Method("ada_only", sigma="adaptive", sil=.012, balance=.0015, rescue=True),
    "author_objective_control": Method("author_objective_control", objective="author_unnormalized", entropy_base="2", temperature=.25),
}


@dataclass(frozen=True)
class Config:
    profile: str = "paper70k"
    split: str = "all70k"
    backbone: str = "author_conv_bn"
    seeds: tuple = tuple(range(10))
    methods: tuple = ("dcss_paper", "dcss_rms", "im_rms", "ada_im")
    pretrain_epochs: int = 100
    cluster_epochs: int = 100
    batch_size: int = 256
    latent_dim: int = 10
    n_clusters: int = 10
    pretrain_lr: float = 1e-3
    cluster_lr: float = 5e-4
    pretrain_weight_decay: float = 1e-5
    kmeans_n_init: int = 100
    grad_clip: float | None = None
    require_cuda: bool = True
    primary_metric: str = "ARI"
    cpu_threads: int = 2


@dataclass(frozen=True)
class RuntimeOptions:
    # Execution settings are deliberately outside the scientific Config/hash.
    device: str = "auto"
    allow_cpu: bool = True
    checkpoint_seconds: float = 300.
    max_session_hours: float = 0.
    allow_software_change: bool = True
    resume_folder: str | None = None


class PauseRequested(RuntimeError):
    """A requested time budget ended at a completed optimizer-step boundary."""


class ResumeCompatibilityError(ValueError):
    pass


def utc_now():
    return datetime.now(timezone.utc).isoformat()


def json_normalize(value):
    return json.loads(json.dumps(value, sort_keys=True))


def hash_json(value):
    return hashlib.sha256(json.dumps(value, sort_keys=True).encode()).hexdigest()


def select_device(runtime):
    if runtime.device not in {"auto", "cuda", "cpu"}:
        raise ValueError("device must be auto/cuda/cpu. TPU requires a separate XLA implementation.")
    if runtime.device != "cpu" and torch.cuda.is_available():
        return torch.device("cuda:0")
    if runtime.device == "cuda" or not runtime.allow_cpu:
        raise RuntimeError("No CUDA GPU available. Select A100/L4/T4 (or another CUDA GPU) in Colab, or enable allow_cpu.")
    print("CPU MODE: same PyTorch model and checkpoint; training will be much slower. GPU selection remains in the Colab UI.", flush=True)
    return torch.device("cpu")


def profile_config(name):
    if name == "paper70k":
        return Config()
    if name in {"legacy70k", "legacy10k"}:
        return Config(profile=name, split="all70k" if name == "legacy70k" else "legacy10k",
                      backbone="legacy_conv", seeds=(0, 1, 2), methods=("dcss_rms", "im_rms", "ada_im"),
                      pretrain_epochs=80, cluster_epochs=40, grad_clip=5.)
    raise ValueError(name)


def seed_all(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


class Autoencoder(nn.Module):
    """Author Conv/BN topology with paper He init, or the legacy local topology.

    Both choices and their differences are recorded in every manifest.
    """
    def __init__(self, latent=10, backbone="author_conv_bn"):
        super().__init__()
        bn = backbone == "author_conv_bn"
        if backbone not in {"author_conv_bn", "legacy_conv"}:
            raise ValueError(backbone)
        enc = []
        for ci, co, k, p in [(1, 32, 5, 2), (32, 64, 5, 2), (64, 128, 3, 0)]:
            enc += [nn.Conv2d(ci, co, k, stride=2, padding=p), nn.ReLU()]
            if bn:
                enc += [nn.BatchNorm2d(co)]
        enc += [nn.Flatten(), nn.Linear(1152, latent), nn.Tanh()]
        if bn:
            enc += [nn.BatchNorm1d(latent)]
        dec = [nn.Linear(latent, 1152), nn.ReLU()]
        if bn:
            dec += [nn.BatchNorm1d(1152)]
        dec += [nn.Unflatten(1, (128, 3, 3))]
        for ci, co, k, p, op in [(128, 64, 3, 0, 0), (64, 32, 5, 2, 1), (32, 1, 5, 2, 1)]:
            dec += [nn.ConvTranspose2d(ci, co, k, stride=2, padding=p, output_padding=op)]
            if co != 1 or bn:
                dec += [nn.ReLU()]
                if bn:
                    dec += [nn.BatchNorm2d(co)]
            else:
                dec += [nn.Sigmoid()]
        self.encoder = nn.Sequential(*enc)
        self.decoder = nn.Sequential(*dec)
        for layer in self.modules():
            if isinstance(layer, (nn.Conv2d, nn.ConvTranspose2d, nn.Linear)):
                nn.init.kaiming_normal_(layer.weight, nonlinearity="relu")
                nn.init.zeros_(layer.bias)
        if not bn:
            nn.init.xavier_uniform_(self.encoder[-2].weight)

    def forward(self, x):
        z = self.encoder(x)
        return self.decoder(z), z


def squared_distances(x, centers):
    # B*K*D is small; avoids sqrt singularity in a Gaussian RBF.
    return (x[:, None, :] - centers[None, :, :]).square().sum(-1)


class RBFHead(nn.Module):
    def __init__(self, centers, embeddings, method):
        super().__init__()
        self.method = method
        self.centers = nn.Parameter(centers.clone())
        self.register_buffer("sigma", torch.ones(len(centers), dtype=centers.dtype))
        self.register_buffer("guard_count", torch.zeros((), dtype=torch.long))
        self.register_buffer("rescue_count", torch.zeros((), dtype=torch.long))
        d2 = squared_distances(embeddings, centers)
        g = d2.mean().clamp_min(1e-6).sqrt()
        if method.sigma == "rms_trainable":
            self.sigma.fill_(g)
        elif method.sigma == "adaptive":
            nearest = d2.argmin(1)
            local = torch.ones_like(self.sigma) * g
            for j in range(len(centers)):
                mask = nearest == j
                if mask.sum() > 2:
                    local[j] = d2[mask, j].mean().clamp_min(1e-6).sqrt()
            self.sigma.copy_((g * (.65 + .35 * local / local.mean().clamp_min(1e-6))).clamp(.55 * g, 1.80 * g))
        elif method.sigma != "one_trainable":
            raise ValueError(method.sigma)
        if method.sigma != "adaptive":
            self.log_sigma = nn.Parameter(self.sigma.log())
        else:
            self.register_parameter("log_sigma", None)

    def active_sigma(self):
        if self.log_sigma is None:
            return self.sigma
        s = self.log_sigma.exp()
        return s.clamp(.15, 4.5) if self.method.sigma == "rms_trainable" else s

    @torch.no_grad()
    def target_sigma(self, d2, q):
        g = d2.mean().clamp_min(1e-6).sqrt()
        mass = q.sum(0).clamp_min(1e-6)
        local = ((q * d2).sum(0) / mass).clamp_min(1e-6).sqrt()
        relative = local / local.mean().clamp_min(1e-6)
        correction = ((1. / q.shape[1]) / (mass / len(q)).clamp_min(1e-4)).pow(.05).clamp(.90, 1.12)
        return (g * (.65 + .35 * relative) * correction).clamp(.55 * g, 1.85 * g)

    def forward(self, z):
        d2 = squared_distances(z, self.centers)
        if self.training and self.method.sigma == "adaptive":
            with torch.no_grad():
                mean_phi = (-d2.detach() / self.sigma.square().clamp_min(1e-8)).clamp(-60., 0.).exp().mean()
                if mean_phi < .08 or mean_phi > .78:
                    q0 = torch.softmax(torch.exp(-d2.detach() / d2.detach().mean().clamp_min(1e-6)) / self.method.temperature, 1)
                    self.sigma.copy_(self.target_sigma(d2.detach(), q0))
                    self.guard_count.add_(1)
        # Clone buffer: later EMA changes cannot invalidate backward's saved tensor.
        s = self.active_sigma().clone()
        phi = torch.exp(-d2 / s.square().clamp_min(1e-8))
        q = torch.softmax(phi / self.method.temperature, 1)
        target = self.target_sigma(d2.detach(), q.detach()) if self.training and self.method.sigma == "adaptive" else None
        return q, target

    @torch.no_grad()
    def after_step(self, target):
        if target is not None:
            self.sigma.lerp_(target, self.method.ema)

    @torch.no_grad()
    def rescue(self, z):
        self.sigma.fill_(squared_distances(z, self.centers).mean().clamp_min(1e-6).sqrt())
        self.rescue_count.add_(1)


def soft_silhouette(z, q, objective="paper_normalized"):
    if len(z) < 2 or q.shape[1] < 2:
        raise ValueError("Soft silhouette requires B>=2, K>=2")
    # Explicit diagonal zero, direct cdist: gradients through both z and q.
    dist = torch.linalg.vector_norm(z[:, None, :] - z[None, :, :], dim=-1)
    dist = dist.masked_fill(torch.eye(len(z), dtype=torch.bool, device=z.device), 0.)
    a = dist @ q
    if objective == "paper_normalized":
        a = a / (q.sum(0, keepdim=True) - q).clamp_min(1e-8)
        k = q.shape[1]
        candidates = a[:, None, :].expand(-1, k, -1).masked_fill(torch.eye(k, dtype=torch.bool, device=z.device)[None], float("inf"))
        b = candidates.min(-1).values
    elif objective == "author_unnormalized":
        # Literal released Objectives.py indexing, preserved only as a diagnostic.
        k = q.shape[1]
        expanded = a.unsqueeze(2).repeat(1, 1, k)
        other = torch.masked_select(expanded, ~torch.eye(k, dtype=torch.bool, device=z.device)[None]).view(len(z), k - 1, k)
        b = other.min(1).values[:, torch.arange(k - 1, -1, -1, device=z.device)]
    else:
        raise ValueError(objective)
    s = (b - a) / torch.maximum(a, b).clamp_min(1e-8)
    return (q * s).sum(1).mean()


def entropy_terms(q):
    h_cond = -(q * q.clamp_min(1e-10).log()).sum(1).mean()
    marginal = q.mean(0)
    kl_uniform = (marginal * (marginal.clamp_min(1e-10) * q.shape[1]).log()).sum()
    return h_cond, kl_uniform


def warmup_weight(epoch, epochs, ratio):
    if ratio == 0:
        return 1.
    progress = epoch / max(epochs - 1, 1)
    t = min(1., max(0., (progress - ratio) / (1. - ratio)))
    return .5 * (1. - math.cos(math.pi * t))


def loss_terms(x, reconstruction, z, q, method, epoch, epochs):
    rec = (x - reconstruction).square().mean()
    sil = soft_silhouette(z, q, method.objective)
    ent, bal = entropy_terms(q)
    ent = ent / math.log(2) if method.entropy_base == "2" else ent
    total = rec + method.sil * (1 - sil) + method.ent * warmup_weight(epoch, epochs, method.warmup) * ent + method.balance * bal
    return total, {"loss": total, "reconstruction": rec, "soft_silhouette": sil, "conditional_entropy": ent, "balance_kl": bal}


def metrics(y, pred):
    _, y = np.unique(y, return_inverse=True)
    _, pred = np.unique(pred, return_inverse=True)
    k = max(y.max(), pred.max()) + 1
    counts = np.zeros((k, k), dtype=np.int64)
    np.add.at(counts, (pred, y), 1)
    row, col = linear_sum_assignment(-counts)
    return {"ACC": float(counts[row, col].sum() / len(y)),
            "NMI": float(normalized_mutual_info_score(y, pred, average_method="arithmetic")),
            "ARI": float(adjusted_rand_score(y, pred))}


def unsupervised(q):
    counts = np.bincount(q.argmax(1), minlength=q.shape[1])
    ent = float(-(q * np.log(np.maximum(q, 1e-10))).sum(1).mean())
    active, largest = int((counts > 0).sum()), float(counts.max() / len(q))
    return {"active_clusters": active, "max_cluster_share": largest, "assignment_entropy": ent,
            "cluster_counts": counts.tolist(),
            "collapsed": bool(active < max(2, int(.3 * q.shape[1])) or largest > .80 or ent > .985 * math.log(q.shape[1]))}


def batch_tensor(images, indices, device):
    x = images[indices].to(device=device, dtype=torch.float32)
    return x / 255. if images.dtype == torch.uint8 else x


@torch.no_grad()
def infer(model, images, device, batch=256, head=None):
    model.eval()
    if head is not None:
        head.eval()
    zs, qs = [], []
    for start in range(0, len(images), batch):
        z = model.encoder(batch_tensor(images, slice(start, start + batch), device))
        zs.append(z.cpu())
        if head is not None:
            qs.append(head(z)[0].cpu())
    return torch.cat(zs), torch.cat(qs) if qs else None


def epoch_batches(n, batch, seed, phase, epoch):
    # Independent deterministic shuffles, identical across compared methods.
    rng = torch.Generator().manual_seed(int(seed) * 1000003 + phase * 100003 + epoch)
    order = torch.randperm(n, generator=rng)
    chunks = list(order.split(batch))
    if len(chunks) > 1 and len(chunks[-1]) == 1:  # BatchNorm requires >=2.
        last = chunks.pop()
        chunks[-1] = torch.cat([chunks[-1], last])
    return chunks


def capture_rng():
    return {"python": random.getstate(), "numpy": np.random.get_state(), "torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else []}


def restore_rng(rng, fallback_seed=0):
    random.setstate(rng["python"])
    np.random.set_state(rng["numpy"])
    torch.set_rng_state(rng["torch"].cpu())
    note = "restored_cpu_rng"
    if torch.cuda.is_available():
        if rng.get("cuda"):
            # The experiment uses one GPU. Restore device 0 even if the number
            # of visible devices changed; never pass nonexistent device indices.
            try:
                torch.cuda.set_rng_state(rng["cuda"][0].cpu(), device=0)
                note = "restored_cpu_and_cuda0_rng"
            except RuntimeError as exc:
                torch.cuda.manual_seed_all(fallback_seed)
                note = "cuda_rng_reseeded_incompatible_state:" + str(exc)
        else:
            torch.cuda.manual_seed_all(fallback_seed)
            note = "cuda_rng_seeded_from_seed_after_cpu_checkpoint"
    elif rng.get("cuda"):
        note = "restored_cpu_rng_cuda_state_not_used_on_cpu"
    return note


def load_checkpoint(path, identity, device="cpu"):
    # Only our own checkpoints, under this run directory. Never load arbitrary pkl.
    obj = torch.load(path, map_location=device, weights_only=False)
    if obj["identity"] != identity:
        raise ValueError("Checkpoint identity differs; refusing to mix protocols/data/code")
    return obj


def cpu_state(value):
    if torch.is_tensor(value):
        return value.detach().cpu().clone()
    if isinstance(value, dict):
        return {k: cpu_state(v) for k, v in value.items()}
    if isinstance(value, list):
        return [cpu_state(v) for v in value]
    if isinstance(value, tuple):
        return tuple(cpu_state(v) for v in value)
    return copy.deepcopy(value)


def save_training_checkpoint(directory, obj):
    """Two immutable-until-next-generation slots with a committed JSON pointer.

    A crash during an inactive-slot write leaves the previous committed slot
    intact. The old v1 checkpoint.pt is never overwritten by this format.
    """
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    index_path = directory / "checkpoint_index.json"
    index = json.loads(index_path.read_text(encoding="utf-8")) if index_path.exists() else {}
    current = index.get("current")
    # If current is damaged but previous is valid, preserve the valid slot.
    for ref in [index.get("current"), index.get("previous")]:
        if ref and (directory / ref["file"]).exists() and digest(directory / ref["file"]) == ref["sha256"]:
            current = ref
            break
    filename = "checkpoint_B.pt" if current and current["file"] == "checkpoint_A.pt" else "checkpoint_A.pt"
    path = directory / filename
    atomic_torch(path, cpu_state(obj))
    ref = {"file": filename, "sha256": digest(path), "next_epoch": obj["next_epoch"],
           "next_batch": obj.get("next_batch", 0), "saved_at": utc_now()}
    atomic_json(index_path, {"schema": 2, "identity": obj["identity"], "current": ref, "previous": current})
    return ref


def load_training_checkpoint(directory, identity):
    directory = Path(directory)
    index_path = directory / "checkpoint_index.json"
    if index_path.exists():
        index = json.loads(index_path.read_text(encoding="utf-8"))
        if index["identity"] != identity:
            raise ResumeCompatibilityError("Checkpoint index identity differs")
        errors = []
        for position, ref in enumerate([index.get("current"), index.get("previous")]):
            if not ref:
                continue
            path = directory / ref["file"]
            if path.parent.resolve() != directory.resolve():
                raise ResumeCompatibilityError("Checkpoint slot outside run directory")
            try:
                if digest(path) != ref["sha256"]:
                    raise IOError("SHA256 mismatch")
                obj = load_checkpoint(path, identity)
                if (obj["next_epoch"], obj.get("next_batch", 0)) != (ref["next_epoch"], ref["next_batch"]):
                    raise ResumeCompatibilityError("Checkpoint progress differs from committed index")
                if position:
                    warnings.warn("Newest checkpoint damaged; recovered previous committed checkpoint: " + str(path))
                obj["_recovered_previous_slot"] = bool(position)
                return obj
            except (OSError, RuntimeError, EOFError) as exc:
                errors.append(f"{path.name}: {exc}")
        raise IOError("No valid committed checkpoint. Refusing to restart from zero: " + "; ".join(errors))
    path = directory / "checkpoint.pt"
    if path.exists():
        return load_checkpoint(path, identity)
    return None


def restore_optimizer(optimizer, state, device):
    optimizer.load_state_dict(state)
    # Adam moments must follow the parameters. Non-capturable Adam keeps its
    # scalar step counter on CPU, including when continuing a CPU checkpoint on CUDA.
    for group in optimizer.param_groups:
        capture = bool(group.get("capturable", False) or group.get("fused", False))
        for parameter in group["params"]:
            for key, value in optimizer.state[parameter].items():
                if torch.is_tensor(value):
                    target = device if key != "step" or capture else torch.device("cpu")
                    optimizer.state[parameter][key] = value.to(target)


def train_phase(model, images, cfg, seed, device, directory, identity, head=None, stop_after=None,
                runtime=None, session=None, stop_after_batches=None):
    """Label-free training; resumable at committed optimizer-step boundaries."""
    runtime = runtime or RuntimeOptions(device=str(device).split(":")[0])
    session = session or {"session_id": "local", "environment": environment(device), "deadline": None}
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    is_cluster = head is not None
    total_epochs = cfg.cluster_epochs if is_cluster else cfg.pretrain_epochs
    parameters = list(model.parameters()) + (list(head.parameters()) if is_cluster else [])
    optimizer = torch.optim.Adam(parameters, lr=cfg.cluster_lr if is_cluster else cfg.pretrain_lr,
                                 weight_decay=0. if is_cluster else cfg.pretrain_weight_decay, betas=(.9, .999))
    start, history, resume_batch, sums, seen, carried_seconds = 0, [], 0, {}, 0, 0.
    saved = load_training_checkpoint(directory, identity)
    if saved is not None:
        model.load_state_dict(saved["model"])
        if is_cluster:
            head.load_state_dict(saved["head"])
        restore_optimizer(optimizer, saved["optimizer"], device)
        rng_note = restore_rng(saved["rng"], seed)
        start, history = saved["next_epoch"], saved["history"]
        resume_batch = saved.get("next_batch", 0)
        sums, seen = saved.get("partial_sums", {}), saved.get("partial_seen", 0)
        carried_seconds = saved.get("partial_seconds", 0.)
        if not 0 <= start <= total_epochs or len(history) != start:
            raise ResumeCompatibilityError("Checkpoint epoch/history are inconsistent")
        atomic_json(directory / ("resume_" + session["session_id"] + ".json"), {
            "session_id": session["session_id"], "at": utc_now(), "epoch": start, "next_batch": resume_batch,
            "rng": rng_note, "previous_environment": saved.get("last_environment"),
            "current_environment": session["environment"], "previous_slot_recovered": saved.get("_recovered_previous_slot", False),
        })
        print(f"Resume {directory.parent.name}/{directory.name}: {start}/{total_epochs} epochs done, next batch {resume_batch}", flush=True)

    trace = list(saved.get("execution_trace", [])) if saved else []
    if saved and not trace:
        trace.append({"session_id": "legacy_v1", "start_epoch": 0, "start_batch": 0,
                      "environment": session.get("legacy_environment", saved.get("last_environment")),
                      "note": "Imported v1 checkpoint; original manifest environment, exact session boundaries unavailable"})
    if start < total_epochs:
        trace.append({"session_id": session["session_id"], "start_epoch": start, "start_batch": resume_batch,
                      "environment": session["environment"]})

    def commit(epoch, batch, partial, count, seconds, peak):
        payload = {"schema": 2, "identity": identity, "next_epoch": epoch, "next_batch": batch,
                   "model": model.state_dict(), "head": head.state_dict() if is_cluster else None,
                   "optimizer": optimizer.state_dict(), "rng": capture_rng(), "history": history,
                   "partial_sums": partial, "partial_seen": count, "partial_seconds": seconds,
                   "partial_peak_gpu_gb": peak, "last_session_id": session["session_id"],
                   "last_environment": session["environment"], "execution_trace": trace}
        save_training_checkpoint(directory, payload)
        atomic_json(directory / "history.json", history)

    if saved is None:
        commit(0, 0, {}, 0, 0., 0.)
    last_commit = time.monotonic()
    steps_this_call = 0
    for epoch in range(start, total_epochs):
        if stop_after is not None and epoch >= stop_after:
            break
        t0 = time.perf_counter()
        if device.type == "cuda":
            torch.cuda.reset_peak_memory_stats(device)
        model.train()
        if is_cluster:
            head.train()
        if epoch != start or resume_batch == 0:
            sums, seen, carried_seconds = {}, 0, 0.
        peak = saved.get("partial_peak_gpu_gb", 0.) if saved and epoch == start and resume_batch else 0.
        batches = epoch_batches(len(images), cfg.batch_size, seed, int(is_cluster), epoch)
        first_batch = resume_batch if epoch == start else 0
        if not 0 <= first_batch <= len(batches) or seen != sum(len(b) for b in batches[:first_batch]):
            raise ResumeCompatibilityError("Checkpoint batch cursor does not match sample count")
        for batch_number in range(first_batch, len(batches)):
            if session.get("deadline") and time.monotonic() >= session["deadline"]:
                commit(epoch, batch_number, sums, seen, carried_seconds + time.perf_counter() - t0, peak)
                raise PauseRequested("Session time budget reached; checkpoint saved. Run all again to continue.")
            indices = batches[batch_number]
            x = batch_tensor(images, indices, device)
            optimizer.zero_grad(set_to_none=True)
            reconstruction, z = model(x)
            target = None
            if is_cluster:
                q, target = head(z)
                loss, terms = loss_terms(x, reconstruction, z, q, head.method, epoch, total_epochs)
            else:
                loss = (x - reconstruction).square().mean()
                terms = {"loss": loss, "reconstruction": loss}
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Non-finite loss at {directory.name}, epoch {epoch}")
            loss.backward()
            norm = nn.utils.clip_grad_norm_(parameters, cfg.grad_clip if cfg.grad_clip is not None and is_cluster else float("inf"), error_if_nonfinite=True)
            optimizer.step()
            if is_cluster:
                head.after_step(target)
            for key, val in terms.items():
                sums[key] = sums.get(key, 0.) + float(val.detach()) * len(x)
            seen += len(x)
            steps_this_call += 1
            if device.type == "cuda":
                peak = max(peak, torch.cuda.max_memory_allocated(device) / 1024**3)
            if time.monotonic() - last_commit >= runtime.checkpoint_seconds or (stop_after_batches is not None and steps_this_call >= stop_after_batches):
                elapsed = carried_seconds + time.perf_counter() - t0
                commit(epoch, batch_number + 1, sums, seen, elapsed, peak)
                last_commit = time.monotonic()
                if stop_after_batches is not None and steps_this_call >= stop_after_batches:
                    return history
        diagnostics = {}
        if is_cluster:
            z_all, q_all = infer(model, images, device, cfg.batch_size, head)
            diagnostics = unsupervised(q_all.numpy())
            if head.method.rescue and epoch in (0, 2) and diagnostics["collapsed"]:
                head.rescue(z_all.to(device))
                diagnostics["rescue_applied_after_epoch"] = True
            diagnostics.update(sigma_min=float(head.active_sigma().min().detach()), sigma_max=float(head.active_sigma().max().detach()),
                               guard_count=int(head.guard_count), rescue_count=int(head.rescue_count))
        elapsed = carried_seconds + time.perf_counter() - t0
        if device.type == "cuda":
            peak = max(peak, torch.cuda.max_memory_allocated(device) / 1024**3)
        row = {"epoch": epoch + 1, **{k: v / seen for k, v in sums.items()}, **diagnostics,
               "seconds": elapsed, "last_batch_grad_norm": float(norm) if first_batch < len(batches) else None, "seen": seen,
               "peak_gpu_gb": peak, "session_id": session["session_id"],
               "epoch_completed_device": str(device), "epoch_completed_gpu": session["environment"].get("gpu"),
               "epoch_resumed_midway": bool(first_batch)}
        history.append(row)
        commit(epoch + 1, 0, {}, 0, 0., 0.)
        last_commit = time.monotonic()
        if epoch == start or (epoch + 1) % 5 == 0 or epoch + 1 == total_epochs:
            print(f"{directory.name} {epoch+1}/{total_epochs}: loss={row['loss']:.5f}, {elapsed:.1f}s/epoch, ~{elapsed*(total_epochs-epoch-1)/3600:.2f}h remaining in this phase", flush=True)
    return history


def environment(device):
    packages = {}
    for name in ["torch", "numpy", "scipy", "scikit-learn", "pandas", "matplotlib", "gdown", "threadpoolctl"]:
        try:
            packages[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            packages[name] = None
    return {"python": platform.python_version(), "platform": platform.platform(), "packages": packages,
            "device": device.type, "cuda": torch.version.cuda, "cudnn": torch.backends.cudnn.version(),
            "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
            "vram_gb": torch.cuda.get_device_properties(device).total_memory / 1024**3 if device.type == "cuda" else None,
            "deterministic_algorithms": True, "amp": False, "tf32": False}


def scientific_config(config):
    config = asdict(config) if isinstance(config, Config) else dict(config)
    # require_cuda/cpu_threads in v1 affected execution, not the mathematical protocol.
    return {k: v for k, v in config.items() if k not in {"require_cuda", "cpu_threads"}}


def mathematical_code_hash(source=None):
    # Hash the AST of all numerical definitions, ignoring formatting/comments.
    # Orchestration and runtime provenance may evolve without changing the science.
    source = Path(__file__).read_text(encoding="utf-8") if source is None else source
    names = {"Autoencoder", "RBFHead", "squared_distances", "soft_silhouette", "entropy_terms",
             "warmup_weight", "loss_terms", "metrics", "unsupervised", "batch_tensor", "infer", "epoch_batches"}
    tree = ast.parse(source)
    nodes = [node for node in tree.body if isinstance(node, (ast.FunctionDef, ast.ClassDef)) and node.name in names]
    if len(nodes) != len(names):
        raise ResumeCompatibilityError("Missing numerical definition in implementation")
    # Also fingerprint the actual Adam constructor, optimization step and
    # epoch-end ADA rescue, while excluding checkpoint/logging orchestration.
    trainer = next(node for node in tree.body if isinstance(node, ast.FunctionDef) and node.name == "train_phase")
    def assigns(node, name):
        return isinstance(node, ast.Assign) and any(isinstance(t, ast.Name) and t.id == name for t in node.targets)
    optimizer = next(node for node in ast.walk(trainer) if assigns(node, "optimizer"))
    loop = next(node for node in ast.walk(trainer) if isinstance(node, ast.For) and any(assigns(s, "x") for s in node.body))
    first = next(i for i, node in enumerate(loop.body) if assigns(node, "x"))
    last = next(i for i, node in enumerate(loop.body) if isinstance(node, ast.AugAssign) and isinstance(node.target, ast.Name) and node.target.id == "seen")
    epoch_loop = next(node for node in ast.walk(trainer) if isinstance(node, ast.For) and any(assigns(s, "diagnostics") for s in node.body))
    diagnostic_pos = next(i for i, node in enumerate(epoch_loop.body) if assigns(node, "diagnostics"))
    nodes += [optimizer, *loop.body[first:last + 1], epoch_loop.body[diagnostic_pos + 1]]
    return hashlib.sha256(ast.dump(ast.Module(body=nodes, type_ignores=[]), include_attributes=False).encode()).hexdigest()


def scientific_spec(cfg, data_meta):
    return json_normalize({"scientific_version": SCIENTIFIC_VERSION, "config": scientific_config(cfg),
                           "methods": [asdict(METHODS[m]) for m in cfg.methods], "data": data_meta,
                           "mathematical_code_sha256": mathematical_code_hash(),
                           "training_semantics": "Adam(.9,.999),fp32,noAMP,noTF32,deterministic,epoch_batches_v1,no_label_selection"})


def make_identity(cfg, data_meta, env):
    spec = scientific_spec(cfg, data_meta)
    identity = hash_json(spec)
    manifest = {"schema": 2, "version": VERSION, "identity": identity, "scientific_spec": spec,
                "config": asdict(cfg), "methods": spec["methods"], "data": data_meta,
                "environment": env, "code_sha256": digest(__file__), "upstream_commit": UPSTREAM_COMMIT}
    return identity, manifest


def compatible_run(root, cfg, data_meta):
    """Return original checkpoint identity if compatible; never select by score."""
    root = Path(root)
    path = root / "manifest.json"
    if not path.exists():
        return None
    manifest = json.loads(path.read_text(encoding="utf-8"))
    spec = scientific_spec(cfg, data_meta)
    portable = root / "portable_resume.json"
    if portable.exists():
        adapter = json.loads(portable.read_text(encoding="utf-8"))
        if adapter["scientific_spec"] != spec:
            return None
        return adapter["checkpoint_identity"], manifest
    if manifest.get("schema") == 2:
        if manifest["scientific_spec"] != spec or manifest["identity"] != hash_json(spec):
            return None
        return manifest["identity"], manifest
    # Import only the exact previously delivered implementation, with identical
    # data, all method coefficients and the full scientific config.
    if manifest.get("code_sha256") != LEGACY_CODE_SHA256:
        return None
    if json_normalize(scientific_config(manifest["config"])) != spec["config"] or manifest["data"] != spec["data"] or manifest["methods"] != spec["methods"]:
        return None
    old_code = root / "experiment.py"
    if not old_code.exists() or digest(old_code) != LEGACY_CODE_SHA256:
        raise ResumeCompatibilityError("v1 source is missing/modified; refusing unverifiable migration: " + str(root))
    if mathematical_code_hash(old_code.read_text(encoding="utf-8")) != spec["mathematical_code_sha256"]:
        raise ResumeCompatibilityError("v1 mathematics differs; cannot resume as the same experiment")
    old_identity = hash_json(manifest)
    # Status is useful but never trusted instead of recomputing the v1 hash.
    status = root / "status.json"
    if status.exists() and json.loads(status.read_text(encoding="utf-8")).get("identity") != old_identity:
        raise ResumeCompatibilityError("v1 status/manifest identity mismatch")
    return old_identity, manifest


def resolve_run(output_root, cfg, data_meta, env, runtime):
    output_root = Path(output_root)
    output_root.mkdir(parents=True, exist_ok=True)
    identity, new_manifest = make_identity(cfg, data_meta, env)
    if runtime.resume_folder:
        root = Path(runtime.resume_folder)
        if not root.is_absolute():
            root = output_root / root
        match = compatible_run(root, cfg, data_meta)
        if match is None:
            raise ResumeCompatibilityError("Selected folder is not compatible: " + str(root))
        identity, manifest = match
    else:
        candidates, incompatible_progress = [], []
        for root in sorted(output_root.glob(cfg.profile + "_*")):
            if not root.is_dir():
                continue
            match = compatible_run(root, cfg, data_meta)
            if match:
                candidates.append((root, *match))
            elif any(root.glob("seed_*/*/checkpoint*.pt")) or any(root.glob("seed_*/*/result.json")):
                incompatible_progress.append(str(root))
        if len(candidates) > 1:
            raise ResumeCompatibilityError("Several compatible runs exist; set RESUME_FOLDERS for this profile instead of mixing them: " + str([str(x[0]) for x in candidates]))
        if candidates:
            root, identity, manifest = candidates[0]
        else:
            if incompatible_progress:
                raise ResumeCompatibilityError("Existing progress has an incompatible config/code/dataset. Refusing a silent restart. Inspect or choose a new OUTPUT_ROOT explicitly: " + str(incompatible_progress))
            root = output_root / (cfg.profile + "_" + identity[:12])
            if root.exists() and any(root.iterdir()):
                raise ResumeCompatibilityError("Target folder has files but no compatible manifest")
            root.mkdir(parents=True, exist_ok=True)
            manifest = new_manifest
            atomic_json(root / "manifest.json", manifest)
            (root / "experiment.py").write_bytes(Path(__file__).read_bytes())
    # Preserve original manifest, source, .pt and scores. Adapter is additive.
    portable_path = root / "portable_resume.json"
    if not portable_path.exists():
        atomic_json(portable_path, {"schema": 2, "created_at": utc_now(), "checkpoint_identity": identity,
                                   "scientific_spec": new_manifest["scientific_spec"], "original_manifest_sha256": digest(root / "manifest.json"),
                                   "migrated_from_v1": manifest.get("schema") != 2})
    else:
        adapter = json.loads(portable_path.read_text(encoding="utf-8"))
        if adapter["original_manifest_sha256"] != digest(root / "manifest.json"):
            raise ResumeCompatibilityError("Original manifest changed after portable resume registration")
    return root, identity, manifest


def start_session(root, original_manifest, device, runtime):
    env = environment(device)
    previous = original_manifest["environment"]
    existing = sorted((root / "sessions").glob("*.json")) if (root / "sessions").exists() else []
    if existing:
        previous = json.loads(existing[-1].read_text(encoding="utf-8"))["environment"]
    software_keys = ("python", "packages", "cuda", "cudnn")
    software_changed = any(previous.get(k) != env.get(k) for k in software_keys)
    hardware_changed = any(previous.get(k) != env.get(k) for k in ("device", "gpu", "vram_gb"))
    if software_changed and not runtime.allow_software_change:
        raise ResumeCompatibilityError("Software changed. Restore the previous runtime version or enable recorded software migration.")
    session_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:8]
    session = {"session_id": session_id, "started_at": utc_now(), "state": "running", "environment": env,
               "previous_environment": previous, "software_changed": software_changed, "hardware_changed": hardware_changed,
               "runtime_options": asdict(runtime), "code_sha256": digest(__file__),
               "cross_environment_bitwise_equivalence_claimed": False}
    atomic_json(root / "sessions" / (session_id + ".json"), session)
    code_archive = root / "implementations" / (digest(__file__) + ".py")
    code_archive.parent.mkdir(exist_ok=True)
    if not code_archive.exists():
        code_archive.write_bytes(Path(__file__).read_bytes())
    if software_changed or hardware_changed:
        print("Continuing saved state on a changed runtime; change recorded. Cross-device bitwise equality is not guaranteed.", flush=True)
    return {**session, "deadline": time.monotonic() + runtime.max_session_hours * 3600 if runtime.max_session_hours > 0 else None,
            "legacy_environment": original_manifest["environment"]}


def checkpoint_trace(directory, identity, legacy_environment):
    checkpoint = load_training_checkpoint(directory, identity)
    if checkpoint is None:
        return []
    return checkpoint.get("execution_trace") or [{"session_id": "legacy_v1", "environment": legacy_environment}]


def run_experiment(images, labels, data_meta, cfg, output_root, runtime=None):
    if len(images) != len(labels) or len(images) != data_meta["n"]:
        raise ValueError("Dataset / labels / metadata length mismatch")
    # Keep CPU-only test callers working; notebook supplies explicit runtime policy.
    runtime = runtime or RuntimeOptions(device="auto" if cfg.require_cuda else "cpu")
    device = select_device(runtime)
    torch.set_num_threads(cfg.cpu_threads)
    seed_all(cfg.seeds[0])
    root, identity, manifest = resolve_run(output_root, cfg, data_meta, environment(device), runtime)
    session = start_session(root, manifest, device, runtime)
    print("RESULT DIRECTORY:", root, flush=True)
    status = {"state": "running", "expected_runs": len(cfg.seeds) * len(cfg.methods), "identity": identity}
    atomic_json(root / "status.json", status)
    try:
        for seed in cfg.seeds:
            seed_all(seed)
            seed_dir = root / f"seed_{seed:02d}"
            done = []
            for name in cfg.methods:
                result = seed_dir / name / "result.json"
                if result.exists():
                    record = json.loads(result.read_text(encoding="utf-8"))
                    if record["identity"] != identity + f":{seed}:{name}":
                        raise ResumeCompatibilityError("Existing result has a different identity")
                    done.append(name)
            if len(done) == len(cfg.methods):
                print(f"Already complete: seed {seed}, all methods. No GPU training needed.", flush=True)
                continue
            model = Autoencoder(cfg.latent_dim, cfg.backbone).to(device)
            train_phase(model, images, cfg, seed, device, seed_dir / "pretrain", identity + f":{seed}:pretrain", runtime=runtime, session=session)
            pretrained = copy.deepcopy(model.state_dict())
            init_path = seed_dir / "initialization.pt"
            init_id = identity + f":{seed}:initialization"
            if init_path.exists():
                init = load_checkpoint(init_path, init_id)
                z_init, centers = init["embeddings"], init["centers"]
            else:
                z_init, _ = infer(model, images, device, cfg.batch_size)
                with threadpool_limits(limits=cfg.cpu_threads):
                    km = KMeans(n_clusters=cfg.n_clusters, n_init=cfg.kmeans_n_init, random_state=seed, algorithm="lloyd").fit(z_init.numpy())
                centers = torch.from_numpy(km.cluster_centers_).float()
                atomic_torch(init_path, {"identity": init_id, "embeddings": z_init, "centers": centers})
            # This baseline is a final evaluation of a fixed pretraining procedure.
            km_pred = squared_distances(z_init, centers).argmin(1).numpy()
            if not (seed_dir / "ae_kmeans_evaluation.json").exists():
                atomic_json(seed_dir / "ae_kmeans_evaluation.json", {"seed": seed, **metrics(labels, km_pred)})
            for method_name in cfg.methods:
                method = METHODS[method_name]
                method_dir = seed_dir / method_name
                result_path = method_dir / "result.json"
                run_id = identity + f":{seed}:{method_name}"
                if result_path.exists():
                    existing = json.loads(result_path.read_text(encoding="utf-8"))
                    if existing["identity"] != run_id:
                        raise ValueError("Result identity mismatch")
                    print(f"Already complete: seed {seed}, {method_name}", flush=True)
                    continue
                seed_all(seed)
                model.load_state_dict(pretrained)
                head = RBFHead(centers, z_init, method).to(device)
                history = train_phase(model, images, cfg, seed, device, method_dir, run_id, head, runtime=runtime, session=session)
                z, q = infer(model, images, device, cfg.batch_size, head)
                q_np, z_np = q.numpy(), z.numpy()
                pred = q_np.argmax(1)
                diagnostics = unsupervised(q_np)
                idx = np.random.default_rng(10000 + seed).choice(len(images), min(2000, len(images)), replace=False)
                unique = np.unique(pred[idx])
                sampled_sil = float(silhouette_score(z_np[idx], pred[idx])) if 1 < len(unique) < len(idx) else None
                trace = checkpoint_trace(seed_dir / "pretrain", identity + f":{seed}:pretrain", manifest["environment"])
                trace += checkpoint_trace(method_dir, run_id, manifest["environment"])
                trace += [{"session_id": session["session_id"], "phase": "final_evaluation", "environment": session["environment"]}]
                environments = {hash_json(x.get("environment")) for x in trace}
                result = {"identity": run_id, "seed": seed, "method": method_name, "profile": cfg.profile,
                          "n": len(images), "epochs": cfg.cluster_epochs, "status": "complete",
                          **metrics(labels, pred), **diagnostics, "sampled_hard_silhouette": sampled_sil,
                          "finetune_seconds": sum(h["seconds"] for h in history), "peak_gpu_gb": max(h["peak_gpu_gb"] for h in history),
                          "guard_count": int(head.guard_count), "rescue_count": int(head.rescue_count),
                          "mixed_environment": len(environments) > 1, "environment_count": len(environments),
                          "execution_trace": trace, "evaluation_environment": session["environment"]}
                # Keep scores, predictions, labels (evaluation only) and embeddings for independent rechecking.
                tmp = method_dir / "predictions.npz.tmp"
                with tmp.open("wb") as f:
                    np.savez_compressed(f, predictions=pred, labels=labels, probabilities=q_np, embeddings=z_np, sample_indices=idx)
                os.replace(tmp, method_dir / "predictions.npz")
                atomic_json(result_path, result)
                print({k: result[k] for k in ("seed", "method", "ARI", "NMI", "ACC", "collapsed")}, flush=True)
                summarize(root, cfg)
                del head
                if device.type == "cuda":
                    torch.cuda.empty_cache()
        status["state"] = "complete"
    except BaseException as exc:
        status.update(state="paused" if isinstance(exc, PauseRequested) else "interrupted_or_failed", error_type=type(exc).__name__, error=str(exc))
        atomic_json(root / "status.json", status)
        summarize(root, cfg)
        raise
    finally:
        report = {k: v for k, v in session.items() if k not in {"deadline", "legacy_environment"}}
        report.update(ended_at=utc_now(), state=status["state"])
        atomic_json(root / "sessions" / (session["session_id"] + ".json"), report)
    atomic_json(root / "status.json", status)
    summarize(root, cfg)
    return root


def paired_statistics(deltas):
    d = np.asarray(deltas, dtype=float)
    n = len(d)
    if n < 2:
        return {"n": n, "mean_delta": float(d.mean()) if n else None, "sd": None, "ci95_low": None, "ci95_high": None, "sign_flip_p_two_sided": None}
    mean, sd = float(d.mean()), float(d.std(ddof=1))
    margin = float(stats.t.ppf(.975, n - 1) * sd / math.sqrt(n))
    if n <= 16:
        signs = np.asarray(list(itertools.product([-1., 1.], repeat=n)))
        p = float(np.mean(np.abs((signs * d).mean(1)) >= abs(mean) - 1e-15))
    else:
        p = None
    return {"n": n, "mean_delta": mean, "sd": sd, "ci95_low": mean - margin, "ci95_high": mean + margin, "sign_flip_p_two_sided": p}


def summarize(root, cfg):
    import pandas as pd
    root = Path(root)
    rows = [json.loads(p.read_text(encoding="utf-8")) for p in sorted(root.glob("seed_*/*/result.json"))]
    if not rows:
        atomic_json(root / "completeness.json", {"completed": 0, "expected": len(cfg.seeds) * len(cfg.methods), "complete": False})
        return
    manifest_path = root / "manifest.json"
    original_env = json.loads(manifest_path.read_text(encoding="utf-8")).get("environment") if manifest_path.exists() else None
    all_envs = set()
    for row in rows:
        trace = row.get("execution_trace") or [{"environment": original_env}]
        known = [t.get("environment") for t in trace if t.get("environment")]
        all_envs.update(hash_json(env) for env in known)
        row["mixed_environment"] = row.get("mixed_environment", False)
        row["environment_provenance"] = "recorded" if row.get("execution_trace") else "legacy_manifest_only" if original_env else "unknown"
    frame = pd.DataFrame(rows)
    campaign_mixed = len(all_envs) > 1 or bool(frame.mixed_environment.any())
    frame.to_csv(root / "per_seed.csv", index=False)
    summaries = []
    for method in cfg.methods:
        group = frame[frame.method == method]
        row = {"method": method, "completed_seeds": len(group), "expected_seeds": len(cfg.seeds),
               "complete": len(group) == len(cfg.seeds), "collapsed_runs": int(group.collapsed.sum()),
               "mixed_environment_runs": int(group.mixed_environment.sum()), "campaign_mixed_environment": campaign_mixed}
        for metric in ("ARI", "NMI", "ACC"):
            row[metric + "_mean"] = float(group[metric].mean()) if len(group) else None
            row[metric + "_sample_sd"] = float(group[metric].std(ddof=1)) if len(group) > 1 else None
        summaries.append(row)
    pd.DataFrame(summaries).to_csv(root / "summary.csv", index=False)
    pairs = []
    for baseline in ("dcss_paper", "dcss_rms", "im_rms"):
        if baseline not in cfg.methods or "ada_im" not in cfg.methods:
            continue
        for metric in ("ARI", "NMI", "ACC"):
            a = frame[frame.method == "ada_im"].set_index("seed")[metric]
            b = frame[frame.method == baseline].set_index("seed")[metric]
            shared = a.index.intersection(b.index).sort_values()
            pairs.append({"comparison": "ada_im - " + baseline, "metric": metric,
                          "paired_seeds": shared.tolist(), "complete": len(shared) == len(cfg.seeds),
                          "planned_primary": cfg.profile == "paper70k" and baseline == "im_rms" and metric == cfg.primary_metric,
                          "confirmatory": cfg.profile == "paper70k" and baseline == "im_rms" and metric == cfg.primary_metric and not campaign_mixed and len(shared) == len(cfg.seeds),
                          "campaign_mixed_environment": campaign_mixed,
                          **paired_statistics((a.loc[shared] - b.loc[shared]).to_numpy())})
    atomic_json(root / "paired_comparisons.json", pairs)
    # Never auto-declare thesis success, particularly from partial/three-seed runs.
    atomic_json(root / "completeness.json", {"completed": len(rows), "expected": len(cfg.seeds) * len(cfg.methods),
                                           "complete": len(rows) == len(cfg.seeds) * len(cfg.methods),
                                           "selection": "fixed_final_epoch_all_seeds_including_collapse", "units": "fractions_not_percent",
                                           "campaign_mixed_environment": campaign_mixed,
                                           "interpretation": "Hardware/software changed: results retained, timings confounded; inspect execution traces before thesis claims." if campaign_mixed else "Inspect all seeds and paired intervals; no automatic success declaration."})


def progress_inventory(output_root):
    """Read small JSON files only, including v1 histories, before GPU work."""
    rows = []
    for manifest_path in sorted(Path(output_root).glob("*/manifest.json")):
        root = manifest_path.parent
        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
        for seed in manifest.get("config", {}).get("seeds", []):
            for phase in ["pretrain", *manifest["config"]["methods"]]:
                directory = root / f"seed_{seed:02d}" / phase
                index, history, result = directory / "checkpoint_index.json", directory / "history.json", directory / "result.json"
                if not any(p.exists() for p in [index, history, result, directory / "checkpoint.pt"]):
                    continue
                record = {"folder": root.name, "seed": seed, "phase": phase, "complete_result": result.exists(), "saved_epochs": 0, "next_batch": 0}
                if index.exists():
                    current = json.loads(index.read_text(encoding="utf-8"))["current"]
                    record.update(saved_epochs=current["next_epoch"], next_batch=current["next_batch"], format="portable_v2")
                elif history.exists():
                    record.update(saved_epochs=len(json.loads(history.read_text(encoding="utf-8"))), format="legacy_v1")
                rows.append(record)
    return rows


def export_review(output_root, roots=None):
    import zipfile
    output_root = Path(output_root)
    roots = [Path(p) for p in roots] if roots is not None else [p.parent for p in output_root.glob("*/manifest.json")]
    path = output_root / "results_for_review.zip"
    temp = path.with_suffix(".zip.tmp")
    with zipfile.ZipFile(temp, "w", compression=zipfile.ZIP_DEFLATED) as archive:
        for shared in ("PROTOCOL_portable_v2.md", "canonical_hashes.json", "verified_dataset.json", "runtime_validation_latest.json", "latest_completed_roots.json", "scale_comparison_paired_seeds.csv"):
            file = output_root / shared
            if file.exists():
                archive.write(file, file.name)
        for root in roots:
            if not root.resolve().is_relative_to(output_root.resolve()):
                raise ValueError("Review archive only includes runs inside OUTPUT_ROOT")
            for file in root.rglob("*"):
                if file.is_file() and file.suffix in {".json", ".csv", ".png", ".py"}:
                    archive.write(file, file.relative_to(output_root))
    os.replace(temp, path)
    return path


def plots(root):
    import pandas as pd
    import matplotlib.pyplot as plt
    root = Path(root)
    frame = pd.read_csv(root / "per_seed.csv")
    names = list(frame.method.unique())
    fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)
    for axis, metric in zip(axes, ("ARI", "NMI", "ACC")):
        for seed, g in frame.groupby("seed"):
            axis.plot([names.index(x) for x in g.method], 100 * g[metric], "o-", alpha=.45, label=str(seed))
        axis.set_xticks(range(len(names)), names, rotation=25, ha="right")
        axis.set_ylabel(metric + " (%)")
        axis.grid(alpha=.2)
    fig.suptitle("All completed seeds; fixed final epoch (check completeness.json)")
    fig.savefig(root / "paired_scores.png", dpi=180)
    plt.show()
    fig, axis = plt.subplots(figsize=(10, 4), constrained_layout=True)
    for path in sorted(root.glob("seed_00/*/history.json")):
        if path.parent.name == "pretrain":
            continue
        history = pd.read_json(path)
        axis.plot(history.epoch, history.soft_silhouette, label=path.parent.name)
    axis.set(xlabel="Epoch", ylabel="Mini-batch soft silhouette (seed 0)")
    axis.legend()
    fig.savefig(root / "training_soft_silhouette.png", dpi=180)
    plt.show()


def technical_smoke(device="cpu", batch_size=16):
    """Small numerical/architecture check, explicitly NOT experimental evidence."""
    seed_all(987654)
    torch.set_num_threads(2)
    device = torch.device(device)
    for backbone in ("author_conv_bn", "legacy_conv"):
        ae = Autoencoder(backbone=backbone).to(device)
        x = torch.rand(batch_size, 1, 28, 28, device=device)
        rec, z = ae(x)
        assert rec.shape == x.shape and z.shape == (batch_size, 10)
        head = RBFHead(z[:10].detach().cpu(), z.detach().cpu(), METHODS["ada_im"]).to(device)
        q, target = head(z)
        loss, _ = loss_terms(x, rec, z, q, head.method, 99, 100)
        loss.backward()
        assert torch.isfinite(loss) and all(p.grad is None or torch.isfinite(p.grad).all() for p in list(ae.parameters()) + list(head.parameters()))
        head.after_step(target)
    print("Technical smoke passed on", device, "(not a research result).")


def portability_smoke(device):
    """Exercise real optimizer migration on the current runtime before long work."""
    import tempfile
    device = torch.device(device)
    technical_smoke(str(device), batch_size=256 if device.type == "cuda" else 16)
    cfg = replace(Config(), require_cuda=False, batch_size=16, pretrain_epochs=1, cluster_epochs=1)
    seed_all(987654)
    images = torch.randint(0, 256, (48, 1, 28, 28), dtype=torch.uint8)
    model = Autoencoder().to(device)
    z, _ = infer(model, images, device, 16)
    centers = z[:10]
    with tempfile.TemporaryDirectory(prefix="dcss_portability_") as folder:
        head = RBFHead(centers, z, METHODS["ada_im"]).to(device)
        train_phase(model, images, cfg, 987654, device, folder, "technical_only", head, stop_after_batches=1)
        model = Autoencoder()
        head = RBFHead(centers, z, METHODS["ada_im"])
        train_phase(model, images, cfg, 987654, torch.device("cpu"), folder, "technical_only", head, stop_after_batches=1)
        model = Autoencoder().to(device)
        head = RBFHead(centers, z, METHODS["ada_im"]).to(device)
        train_phase(model, images, cfg, 987654, device, folder, "technical_only", head)
        final = load_training_checkpoint(folder, "technical_only")
        assert final["next_epoch"] == 1 and final["next_batch"] == 0
        assert all(value.device.type == "cpu" for value in final["model"].values())
    return {"type": "technical_only_not_research", "environment": environment(device), "at": utc_now(),
            "batch256_gpu_check": device.type == "cuda", "optimizer_roundtrip": str(device) + " -> cpu -> " + str(device),
            "passed": True}


if __name__ == "__main__":
    technical_smoke()


In [ ]:
import importlib, sys
if "/content" not in sys.path:
    sys.path.insert(0, "/content")
import dcss_full_experiment as exp
importlib.reload(exp)
EXPECTED_CODE_SHA256 = 'e7fcbcd69e0ee7ed572f102f1c50f89bd3c3c0d805dbd583ae4c4c022560b550'
assert exp.digest(exp.__file__) == EXPECTED_CODE_SHA256, "Mã thực thi bị thay đổi so với notebook đã chốt."
CANONICAL_HASHES = {'train-images-idx3-ubyte': 'c59f468a2f672dc815687fe0f83887768d799fd8a3f3276145d20f83aa44d888', 'train-labels-idx1-ubyte': 'bad3541b69d912435c50bb6ba87bec294ff4f6a2e1246121d8633921760443d9', 't10k-images-idx3-ubyte': '5b4141f0afbad91edebe8549f8fcffe087ea10ca49f1dbef5c9a5cd8815ce37b', 't10k-labels-idx1-ubyte': '0402a96d92fd2663957122ceb108a494c5af83dab82d92729df917d7dec38c34'}
PROTOCOL_TEXT = '# Giao thức kiểm chứng DCSS và ADA-IM-DCSS trên Fashion-MNIST\n\nGiao thức khoa học chốt ngày 09/11/2026; cập nhật cách vận hành ngày 10/11/2026\nđể tiếp tục checkpoint T4 trên A100, GPU CUDA khác hoặc CPU. Bản cập nhật không\nthay đổi dữ liệu, cấu trúc mạng, batch, hàm mục tiêu, số epoch hay seed. Tệp này\nmô tả những gì chương trình thực hiện và giới hạn của kết luận được phép rút ra.\n\n## 1. Câu hỏi và phạm vi kết luận\n\nMục tiêu thứ nhất là đối chiếu DCSS trên đủ 70.000 ảnh với bài báo.\nMục tiêu thứ hai là kiểm tra ADA-IM-DCSS có duy trì lợi ích khi tăng quy mô dữ\nliệu, đồng thời xác định thành phần ADA có đem lại lợi ích so với chỉ dùng IM.\nHai mục tiêu cần các phép so sánh riêng, vì thay đổi đồng thời kiến trúc, số\nepoch và số ảnh sẽ không cho biết yếu tố nào gây ra thay đổi kết quả.\n\nNotebook mặc định chạy ba hồ sơ cấu hình, theo thứ tự:\n\n| Hồ sơ | Ảnh | Kiến trúc | Tiền huấn luyện / phân cụm | Seed | Phương pháp |\n|---|---:|---|---:|---|---|\n| `paper70k` | 70.000 | Conv/BN theo mã tác giả, khởi tạo He theo bài báo | 100 / 100 | 0–9 | DCSS theo phương trình; DCSS-RMS; IM-DCSS; ADA-IM-DCSS |\n| `legacy10k` | 10.000 ảnh test chính thức | Conv không BN, đầu ra sigmoid theo bản cục bộ | 80 / 40 | 0, 1, 2 | DCSS-RMS; IM-DCSS; ADA-IM-DCSS |\n| `legacy70k` | 70.000 | Giống `legacy10k` | 80 / 40 | 0, 1, 2 | Giống `legacy10k` |\n\nTổng cộng 58 lượt phân cụm và 16 lượt tiền huấn luyện dùng chung. Đây là một\nchiến dịch nhiều lượt, có thể cần nhiều phiên Colab. Nhật ký cũ cho thấy đã\nkhởi chạy trên T4, nhưng chưa có bảng thời lượng toàn chiến dịch được xác minh.\nChương trình chỉ ước lượng thời gian còn lại của pha hiện tại sau khi\nđã đo epoch. Không hạ số epoch/seed âm thầm để hoàn thành trong một phiên.\n\n`legacy10k` được chạy mới bằng cùng bộ thực thi với `legacy70k`. Nó kế thừa\nkiến trúc, siêu tham số và hàm mục tiêu cũ, nhưng lịch trộn mẫu, việc bỏ nhánh\nresidual không dùng và trạng thái ngẫu nhiên khác bộ thực thi cũ. Vì vậy không\nđòi hỏi tái tạo đúng từng chữ số của bảng kết quả cũ. Hai lượt chạy mới mới là\ncặp phù hợp để đánh giá tác động của tăng quy mô dữ liệu.\n\nTập 70.000 được nối theo thứ tự **train 60.000 → test 10.000**. Huấn luyện\nkhông dùng nhãn; nhãn chỉ kiểm tra dữ liệu và tính chỉ số sau huấn luyện.\nĐây là phân cụm chuyển dẫn (*transductive clustering*), không phải thử nghiệm\nkhái quát hóa trên tập kiểm tra độc lập. Hơn nữa, 10.000 ảnh đã dùng trước đây\nđể phát triển phương pháp nằm trong bộ 70.000; không được gọi bộ 70.000 là tập\nchưa từng được sử dụng trong quá trình nghiên cứu.\n\n## 2. Đối chiếu bài báo, mã tác giả và lựa chọn triển khai\n\nNguồn chính:\n\n- [DCSS, Machine Learning 115:81, 2026](https://link.springer.com/article/10.1007/s10994-026-07026-w), phương trình (13)–(22), mục 5.3 và 5.5, bảng 2.\n- [Mã tác giả tại commit cố định](https://github.com/gvardakas/Soft-Silhouette/tree/fa3d84c0f3e7c682729cd5ba6f856bb3ef532637).\n- [RIM: Discriminative Clustering by Regularized Information Maximization, NeurIPS 2010](https://proceedings.neurips.cc/paper/2010/hash/42998cf32d552343bc8e460416382dca-Abstract.html).\n- [SHOT, ICML 2020, phương trình (3)](https://proceedings.mlr.press/v119/liang20a/liang20a.pdf).\n- [Fashion-MNIST chính thức](https://github.com/zalandoresearch/fashion-mnist).\n\nĐường dẫn RIM được cung cấp ban đầu có mã băm khác; liên kết phía trên là trang\ncủa đúng bài của Krause, Perona và Gomes. RIM cung cấp cơ sở tối đa hóa thông\ntin tương hỗ cho phân cụm; SHOT sử dụng entropy có điều kiện và entropy biên\ntrong thích nghi miền. Không xem RIM/SHOT là chứng minh cho các hệ số, lịch\nwarmup hay quy tắc băng thông ADA cụ thể của đề xuất này.\n\n| Nội dung | Bài báo / mã công bố | Quyết định trong `paper70k` |\n|---|---|---|\n| Soft silhouette | (13) chia khoảng cách có trọng số cho tổng xác suất, loại mẫu đang xét khỏi mẫu số. `Objectives.py` công bố thiếu phép chia này và có cách xếp chỉ số riêng. | Theo (13)–(17), có kiểm tra bằng phép tính vòng lặp độc lập và kiểm tra gradient. |\n| Softmax RBF | Bài báo: nhân hệ số T=20. `forward_softMax` công bố mặc định chia 0,25, tương đương T=4. | Chia 0,05, tương đương T=20. |\n| Băng thông | Bài báo: giá trị dương nhỏ, tham số được học; không chỉ rõ trị số. Mã công bố khởi tạo log_sigma=0. | `dcss_paper`: sigma=1 rồi học log_sigma. Không gọi đây là sigma cố định. |\n| Kiến trúc ảnh | Mã tác giả có BatchNorm sau các khối và sau tanh, kể cả đầu ra giải mã. Bản cục bộ không có BN, đầu ra sigmoid. | Dùng cấu trúc Conv/BN của mã tác giả cho cả bốn phương pháp; không trộn hai kiến trúc trong cùng hồ sơ. |\n| Khởi tạo mạng | Bài báo nêu He; mã lớp công bố không có lời gọi He rõ ràng. | He cho Conv/Linear; lựa chọn được ghi trong manifest. |\n| MSE | Công thức mô tả tổng sai số trên chiều; mã tác giả dùng `nn.MSELoss` lấy trung bình cả pixel. | Trung bình pixel theo mã, tránh thay tỷ lệ 784 lần mà không công bố. |\n| Entropy | Bài báo không nêu cơ số log một cách tường minh; mã dùng cơ số 2. | Log tự nhiên, trung bình mẫu, theo công thức thường dùng cho IM và bản cục bộ. Chưa coi tương đương hệ số với entropy cơ số 2. |\n| Tiền xử lý | Không có bộ nạp Fashion-MNIST đầy đủ trong các tệp tác giả đã kiểm tra; loader EMNIST có MinMaxScaler. | Pixel / 255, không tăng cường ảnh, không chuẩn hóa từng ảnh. Đây là lựa chọn triển khai được công bố. |\n| Đánh giá và BN | Mã công bố thu dự đoán trong quá trình cập nhật theo batch. | Một lượt `eval()` sau epoch cuối; không cập nhật BN/ADA khi đánh giá. |\n\nVì các bất nhất trên, tên đúng của kết quả là **tái triển khai DCSS theo phương\ntrình với các lựa chọn được công bố**, chưa phải tái lập nguyên trạng chương\ntrình đã sinh bảng 2. Nếu kết quả lệch bảng 2, cần đối chiếu những lựa chọn này\nvới tác giả trước khi kết luận phương pháp gốc thất bại. Không hứa đạt lại\nđiểm công bố. Bảng 2 báo cáo Fashion-MNIST NMI 0,63 ± 0,02 và ARI 0,45 ± 0,03;\nđây chỉ là mốc tham khảo, không được đưa vào cột kết quả thực nghiệm mới.\n\nCó thêm `author_objective_control` trong mã để chẩn đoán khác biệt: giữ biểu\nthức/indexing soft silhouette của mã công bố, entropy cơ số 2 và temperature\n0,25. Đây vẫn là phép kiểm soát hàm mục tiêu trên bộ thực thi chung, **không**\nđược ghi là chạy nguyên xi toàn bộ repository tác giả. Biến thể này tắt mặc\nđịnh; nếu bật phải tạo hồ sơ mới, không thay thế kết quả chính sau khi xem điểm.\n\n## 3. Hàm mục tiêu và nội dung đề xuất\n\nVới batch B ảnh, K=10, q là xác suất phân cụm:\n\n```text\na_ik = sum_{j != i} q_jk ||z_i-z_j|| / max(sum_{j != i} q_jk, epsilon)\nb_ik = min_{l != k} a_il\nS    = mean_i sum_k q_ik (b_ik-a_ik) / max(a_ik,b_ik,epsilon)\nq_ik = softmax_k(exp(-||z_i-c_k||² / sigma_k²) / 0.05)\nHc   = -mean_i sum_k q_ik ln(q_ik)\nqbar = mean_i q_i\nKL   = sum_k qbar_k ln(K*qbar_k)\n```\n\nDCSS: `L = MSE + 0.01*(1-S) + 0.01*Hc`.\n\nIM-DCSS và ADA-IM-DCSS:\n`L = MSE + 0.012*(1-S) + 0.0025*omega(t)*Hc + 0.004*KL`.\n`omega=0` trong 70% đầu số epoch phân cụm, sau đó tăng theo nửa đường cosine\nđến 1 ở epoch cuối. KL nhỏ khuyến khích phân bố cụm cân bằng; Hc nhỏ khuyến\nkhích gán cụm dứt khoát. Với hệ số bằng nhau và không warmup, tổng hai thành\nphần entropy tương đương âm thông tin tương hỗ cộng hằng số. Ở cấu hình hiện\ntại, cần gọi là **điều chuẩn IM có trọng số và lịch điều chỉnh**, không gọi là\nđúng nguyên dạng hàm âm thông tin tương hỗ của RIM/SHOT.\n\nADA giữ cơ chế đang có: tâm cụm học bằng gradient; sigma khởi tạo theo phân\ntán từng cụm, cập nhật bằng EMA hệ số 0,15; mục tiêu sigma kết hợp RMS toàn cục,\nRMS nội cụm mềm và hiệu chỉnh khối lượng cụm nhẹ. Ngưỡng bảo vệ trung bình RBF\nlà 0,08 / 0,78. Sau epoch 1 và 3, chỉ nếu tiêu chí không nhãn phát hiện suy biến\nthì đặt lại sigma bằng RMS. Mọi lần bảo vệ/phục hồi đều được đếm và ghi lại.\nEMA dùng thống kê trước bước tối ưu nhưng thực hiện cập nhật buffer sau\n`backward/step`, để tránh sửa trạng thái đang được autograd sử dụng.\n\n`normalize_latent_scale` trong cấu hình cũ chỉ được lưu mà không có phép chuẩn\nhóa thực sự trong forward. Bản này không tuyên bố đã thêm chuẩn hóa latent.\nCác ngưỡng, warmup và hệ số ADA được giữ từ bản cũ; chưa có chứng cứ rằng đó là\ncấu hình tối ưu hoặc là đóng góp mới đã được xác nhận bởi tổng quan tài liệu.\n\nDCSS-RMS là đối chứng cần thiết: nó cũng khởi tạo sigma theo RMS rồi học bằng\ngradient, giới hạn [0,15; 4,5], nhưng không có EMA/guard/rescue. Nhờ đối chứng\nnày, việc thắng DCSS sigma khởi tạo 1 không bị hiểu nhầm thành bằng chứng toàn\nbộ lợi ích đến từ ADA. Đặc biệt, ADA-IM so với IM-DCSS giữ nguyên các hệ số hàm\nmục tiêu; cặp này kiểm tra đóng góp của **cả gói ADA** gồm khởi tạo, EMA, guard\nvà rescue. Chưa cô lập riêng từng thành phần của gói ADA. So DCSS với IM còn\nthay đổi hệ số silhouette/entropy, nên chưa phải phép loại bỏ thành phần IM\nthuần túy. `ada_only` có sẵn như một đối chứng bổ sung, tắt mặc định.\n\n## 4. Kiểm soát thực nghiệm và tài nguyên\n\nMỗi seed tiền huấn luyện đúng một lần, lấy K-means với `n_init=100`, rồi chia\ncùng trọng số và tâm khởi tạo cho các phương pháp của seed đó. Thứ tự batch\ncũng giống nhau giữa các phương pháp. Adam dùng beta=(0,9; 0,999), LR tiền\nhuấn luyện 1e-3 và weight decay 1e-5; LR phân cụm 5e-4, không weight decay.\nBatch 256, latent 10. Hồ sơ paper không cắt gradient; hồ sơ legacy cắt norm\n5 ở pha phân cụm theo bản cũ. Không early stopping, không chọn seed tốt nhất,\nkhông chọn checkpoint theo nhãn thật, không điều chỉnh hệ số từ kết quả 70k.\n\nẢnh lưu uint8 trên RAM máy; chỉ batch hiện tại được đổi float32 và chuyển GPU.\nSoft silhouette tính trong batch, không tạo ma trận 70.000 × 70.000. Khoảng\ncách N × K chỉ dùng khi khởi tạo/chẩn đoán tâm; đánh giá encoder cũng theo\nbatch. FP32, AMP/TF32 tắt; bật thuật toán xác định. Nếu CUDA báo thao tác không\nhỗ trợ chế độ xác định, chương trình dừng thay vì tự đổi chế độ. Mỗi epoch lưu\nthời gian, mức sử dụng VRAM đỉnh của PyTorch và chẩn đoán không nhãn.\n\nGiữ bản torch/CUDA đang cài trên Colab. Không ép hạ cấp hoặc cài lại torch,\ntorchvision. Lưu phiên bản Python, torch, CUDA, cuDNN, GPU và thư viện tính\ntoán thực tế. A100 là lựa chọn ưu tiên trong giao diện, không phải yêu cầu cứng\ntrong mã: dùng GPU CUDA mà Colab thực sự cấp. Không tăng batch hoặc bật AMP/TF32\nchỉ vì A100 mạnh hơn, vì việc đó sẽ đổi thí nghiệm đang tiếp tục. RAM cao cấp\nkhông bắt buộc với cách nạp ảnh uint8/batch hiện tại.\n\n`DEVICE="auto"` ưu tiên CUDA, nếu không có thì `ALLOW_CPU_FALLBACK=True` cho\nchạy cùng mạng PyTorch trên CPU. Đây không phải mô hình NumPy thay thế. CPU có\nthể rất chậm; đặt `ALLOW_CPU_FALLBACK=False` nếu chỉ muốn chạy khi có GPU.\nKhông gọi một lượt có đoạn CPU là lượt chạy thuần A100. TPU cần bộ thực thi\nXLA riêng và chưa được hỗ trợ. Khi Colab thu hồi runtime, notebook không thể\ntự xin một GPU khác hoặc vượt hạn mức thuê bao; người dùng chọn thiết bị mới\nrồi Run all. Thông tin giới hạn và GPU phụ thuộc tài nguyên của Colab:\n[FAQ chính thức](https://research.google.com/colaboratory/faq.html#resource-limits).\n\n## 5. Đánh giá đã định trước\n\nChỉ số chính: **ARI**. Giả thuyết chính của phần cải tiến là chênh lệch ARI\n`ADA-IM − IM-DCSS` trên 10 seed của `paper70k`. ACC (ghép nhãn bằng Hungarian\nchính xác) và NMI (trung bình số học) là chỉ số phụ. So với DCSS-paper và\nDCSS-RMS là đối chiếu bổ sung; không dùng nhiều phép so sánh rồi chỉ báo cáo\nphép có lợi. Nếu muốn kết luận xác nhận cho nhiều giả thuyết thì cần hiệu\nchỉnh đa kiểm định trong một giao thức bổ sung.\n\nBản vận hành V2 cho tiếp tục qua nhiều loại GPU/phần mềm để không bỏ công đã\nchạy. Vì thế, ngoài chỉ số chất lượng phải kiểm tra `execution_trace`,\n`mixed_environment` và `campaign_mixed_environment`. Cột `planned_primary`\ngiữ giả thuyết chính đã chốt; cột `confirmatory` chỉ bật khi đủ cặp seed và\nkhông phát hiện pha trộn môi trường. Kết quả có đổi môi trường vẫn giữ đầy đủ,\nnhưng phép suy luận mang tính thăm dò cho đến khi đánh giá được ảnh hưởng của\nthay đổi đó. Không so trực tiếp thời gian tổng của lượt T4/A100/CPU pha trộn để\nkhẳng định thuật toán nhanh hơn. Nếu muốn kết luận tốc độ hoặc cải tiến nhỏ,\nnên bổ sung các cặp chạy trong cùng điều kiện, báo cáo riêng; không xóa lượt cũ.\n\nBáo cáo mọi seed gồm cả suy biến: trung bình, độ lệch chuẩn mẫu, chênh lệch\ntheo cặp seed, khoảng tin cậy t 95%, kiểm định đổi dấu chính xác hai phía.\nKhoảng tin cậy mô tả biến thiên do seed trên tập dữ liệu cố định, không phải\nđộ bất định khi lấy mẫu một quần thể ảnh mới. Với 3 seed legacy, chỉ số thống\nkê mang tính thăm dò; không đủ để tuyên bố cải tiến vững chắc.\n\nKhông kết luận ADA có lợi chỉ vì trung bình cao hơn một chút. Cần xem khoảng\ntin cậy chênh lệch chính, tỷ lệ suy biến, từng seed, thời gian và VRAM. Nếu ADA\nkhông hơn IM, cần báo cáo đúng và xem lại độ cần thiết của thành phần ADA.\nNếu chưa đủ seed, báo cáo là **chưa hoàn tất**. Không loại lượt chạy kém, không\nghi lượt lỗi thành điểm 0 hoặc bỏ qua lỗi mà vẫn coi chiến dịch thành công.\n\n## 6. Dữ liệu và bằng chứng đầu ra\n\nBốn tệp Drive được so byte bằng SHA-256 với bản giải nén từ kho Fashion-MNIST\nchính thức; các bản nén chính thức được kiểm tra MD5 theo bảng phát hành.\n`audit/dataset_verification.json` ghi kết quả so khớp, magic/header/chiều dài\nIDX và số lượng lớp. Mỗi lần chạy kiểm tra lại SHA-256, số ảnh và thứ tự nối.\n\nMỗi thư mục kết quả có `manifest.json`, bản mã `experiment.py`, `status.json`,\n`completeness.json`, `per_seed.csv`, `summary.csv`, `paired_comparisons.json`.\nTrong từng seed có trọng số tiền huấn luyện, tâm K-means dùng chung; mỗi phương\npháp có checkpoint, `history.json`, `result.json`, `predictions.npz` gồm dự\nđoán, xác suất, embedding và nhãn phục vụ kiểm tra độc lập. Đơn vị chỉ số trong\nCSV/JSON là tỷ lệ 0–1, khi vẽ mới nhân 100. ARI có thể âm.\n\nCheckpoint V2 lưu model, buffer BN/ADA, optimizer, RNG, epoch/batch tiếp theo,\ntổng loss và số mẫu của phần epoch đã chạy, thời gian tích lũy và lịch sử thiết\nbị. Lưu ở cuối mỗi epoch và sau một bước tối ưu khi quá 300 giây kể từ lần\nlưu trước. Phiên kết thúc đột ngột sẽ chạy lại phần sau checkpoint gần nhất;\nkhông bảo đảm lưu được bước đang thực hiện ngay lúc Colab thu hồi máy.\n\nHai slot `checkpoint_A.pt` và `checkpoint_B.pt` ghi luân phiên bằng tệp tạm.\n`checkpoint_index.json` trỏ tới bản đã ghi xong kèm SHA-256, giữ tham chiếu bản\ntrước. Nếu bản mới hỏng, thử phục hồi bản trước và thông báo. Nếu cả hai không\nđọc được, chương trình dừng để xử lý, không tự bắt đầu lại. Cơ chế này giảm\nrủi ro khi ghi gián đoạn, không thay thế sao lưu hoặc bảo đảm tuyệt đối cho\nhệ thống tệp Drive. Chỉ chạy một phiên ghi cùng thư mục tại một thời điểm.\n\nĐịnh danh V2 gắn với cấu hình khoa học, dữ liệu, các hệ số phương pháp và mã\nAST của các phép tính cốt lõi; GPU/CPU, hệ điều hành và phiên bản thư viện nằm\ntrong nhật ký riêng. Đổi thiết bị không tạo lại thí nghiệm. Đổi batch, kiến\ntrúc, loss, số epoch hoặc dữ liệu không được phép tiếp tục chung. Nếu tìm\nthấy tiến độ khác cấu hình, chương trình yêu cầu xử lý rõ ràng thay vì âm thầm\ntạo lượt chạy mới. Nếu có nhiều thư mục tương thích, chỉ định `RESUME_FOLDERS`;\nkhông lựa chọn theo điểm hoặc ghép seed từ các thư mục khác nhau.\n\nĐối với dữ liệu đã chạy bằng V1, chỉ tự nhận bản mã có SHA-256\n`b29c6c15ccc41ac03adea4edca8527f9971c70ab923c291d81c889421bbe5703`, cấu hình và\ndữ liệu khớp hoàn toàn, mã toán học khớp V2. Mã tính lại định danh cũ từ\nmanifest, giữ nguyên thư mục, `manifest.json`, `experiment.py`, checkpoint V1\nvà kết quả đã xong; thêm `portable_resume.json` để ghi ánh xạ tương thích.\nCác checkpoint mới dùng hai slot V2. Notebook liệt kê tiến độ trước huấn luyện\nđể người dùng phát hiện việc gắn nhầm tài khoản Drive. Công cụ kết nối trong\nphiên chuẩn bị chưa tìm thấy thư mục kết quả qua tìm kiếm Drive; vì vậy không\ntuyên bố đã xác minh trực tiếp các checkpoint T4 thật của người dùng. Phép\nkiểm chứng nhập V1 được làm trên checkpoint tạo bằng mã V1 đã lưu nguyên bản.\n\nTrạng thái được đọc về CPU trước, sau đó đưa model và các moment của Adam lên\nthiết bị đích; bộ đếm Adam không capturable giữ trên CPU. Đây là cách tiếp tục\ncả model lẫn optimizer, không phải chỉ tải trọng số rồi tạo optimizer mới.\nXem [hướng dẫn checkpoint và chuyển thiết bị của PyTorch](https://docs.pytorch.org/tutorials/beginner/saving_loading_models.html).\n\nMỗi lần tiếp tục tạo một mục `sessions/`, bản mã trong `implementations/` và\ndấu mốc ở từng pha. `ALLOW_SOFTWARE_CHANGE=True` cho phép thử tiếp tục khi\nColab đổi phiên bản, với nhật ký thay đổi; nếu định dạng hoặc toán tử không\ntương thích vẫn dừng có thông báo. Đặt False để yêu cầu khớp phần mềm. Khôi\nphục RNG CUDA trên thiết bị 0 nếu hợp lệ; checkpoint CPU không có RNG CUDA\nthì khởi tạo lại RNG GPU theo seed và ghi rõ. Mạng hiện tại không có dropout,\nthứ tự batch được tạo độc lập trên CPU theo seed/epoch.\n\nTính tiếp tục của trạng thái không đồng nghĩa với tái tạo từng bit. PyTorch\nkhông bảo đảm tái lập tuyệt đối giữa thiết bị, nền tảng hoặc phiên bản:\n[tài liệu về khả năng tái lập](https://docs.pytorch.org/docs/2.15/notes/randomness.html).\nNotebook thực hiện kiểm tra ngắn chuyển optimizer thiết bị hiện tại → CPU →\nthiết bị hiện tại trước khi chạy dài, đồng thời thử batch 256 trên GPU. Đây là\nkiểm tra kỹ thuật của đúng runtime được cấp, không phải số liệu chất lượng.\n\n`MAX_SESSION_HOURS=0` không giới hạn thêm ngoài Colab. Giá trị dương cho phép\ntạm dừng tại ranh giới bước tối ưu, lưu checkpoint và xuất kết quả tạm. Dừng\nbằng nút Stop hoặc mất phiên sử dụng checkpoint đã ghi trước đó. NaN/hết bộ\nnhớ không được xử lý bằng cách tự giảm batch. Phiên mới chọn GPU khác hoặc\nCPU rồi Run all; các lượt đã hoàn tất bỏ qua cả bước dựng/pretrain mạng.\n\n## 7. Những việc còn cần trước khi đưa vào luận văn\n\nChưa xác minh được kết quả GPU đầy đủ ở lần cập nhật này. Đã quan sát lỗi\nPyTorch CPU của notebook cũ; chưa chạy A100 trực tiếp trong phiên chuẩn bị.\nCần chạy\nđủ các hồ sơ, kiểm tra `complete=true`, rồi phân tích chênh lệch. Bất nhất giữa\nbài báo và mã tác giả cần được trình bày trung thực trong phần tái lập; có thể\nxin tác giả xác nhận tiền xử lý, sigma, cơ số entropy, MSE, BN và mã đã sinh\nbảng kết quả. Sau đó mới quyết định có cần một lượt tái lập bổ sung.\n\nKết quả trên một bộ dữ liệu chưa đủ khẳng định phương pháp tổng quát hay tính\nmới. Bài toán mất cân bằng lớp, các bộ ảnh khác và phân tích riêng từng thành\nphần ADA là những kiểm chứng tiếp theo, không được coi là đã hoàn thành chỉ\nbằng Fashion-MNIST cân bằng.\n'
if "OUTPUT_ROOT" not in globals():
    raise RuntimeError("Chưa chạy ô 2 để gắn Drive/cấu hình. Chọn Run all từ đầu; không chạy riêng ô này.")
(OUTPUT_ROOT / "PROTOCOL_portable_v2.md").write_text(PROTOCOL_TEXT, encoding="utf-8")
exp.atomic_json(OUTPUT_ROOT / "canonical_hashes.json", CANONICAL_HASHES)
print("Code SHA256:", EXPECTED_CODE_SHA256)
runtime_policy = exp.RuntimeOptions(device=DEVICE, allow_cpu=ALLOW_CPU_FALLBACK,
    checkpoint_seconds=CHECKPOINT_SECONDS, max_session_hours=MAX_SESSION_HOURS,
    allow_software_change=ALLOW_SOFTWARE_CHANGE)
active_device = exp.select_device(runtime_policy)
print("Thiết bị dùng:", active_device)
from IPython.display import display
import pandas as pd
inventory = exp.progress_inventory(OUTPUT_ROOT)
if inventory:
    print("Tiến độ đã lưu trên Drive (chưa tải lại model):")
    display(pd.DataFrame(inventory))
else:
    print("Chưa tìm thấy tiến độ trong OUTPUT_ROOT. Nếu đã chạy T4, kiểm tra đúng tài khoản Drive/thư mục trước khi chạy ô 5.")
runtime_validation = exp.portability_smoke(active_device)
exp.atomic_json(OUTPUT_ROOT / "runtime_validation_latest.json", runtime_validation)
print("Kiểm tra checkpoint/optimizer trên runtime hiện tại đã qua; đây không phải số liệu luận văn.")

## 4. Tải và xác minh đủ 70.000 ảnh
Ưu tiên bốn tệp trong Drive đã cung cấp. Nguồn thay thế chính thức, nếu cần, phải khớp SHA-256. Sai header, thiếu ảnh hoặc sai checksum đều dừng.

In [ ]:
exp.prepare_data(DATA_ROOT, CANONICAL_HASHES)
images70, labels70, metadata70 = exp.load_data(DATA_ROOT, CANONICAL_HASHES, "all70k")
print("Shape:", tuple(images70.shape), "dtype:", images70.dtype)
print("Số mẫu mỗi lớp (chỉ kiểm tra dữ liệu):", metadata70["class_counts_audit_only"])
assert len(images70) == 70000
exp.atomic_json(OUTPUT_ROOT / "verified_dataset.json", metadata70)
import matplotlib.pyplot as plt
import numpy as np
fig, axes = plt.subplots(2, 10, figsize=(14, 3))
sample_ids = np.random.default_rng(20261109).choice(len(images70), 20, replace=False)
for axis, index in zip(axes.ravel(), sample_ids):
    axis.imshow(images70[index, 0], cmap="gray")
    axis.axis("off")
fig.suptitle("Ảnh kiểm tra dữ liệu; nhãn không đưa vào huấn luyện")
plt.show()

## 5. Chạy thực nghiệm đã chốt

Chỉ ô này thực hiện huấn luyện đầy đủ. Sau mỗi seed/phương pháp có kết quả
cuối; sau mỗi epoch và định kỳ giữa epoch có checkpoint. Không chọn điểm tốt nhất theo nhãn.
Phải chờ `complete=true` cho từng hồ sơ. Nếu lỗi/NaN/hết VRAM, ô dừng và lưu
trạng thái lỗi; không tự thay đổi batch hay dùng thuật toán thay thế.

Khi tiếp tục, kiểm tra `RESULT DIRECTORY` và dòng `Resume ... next batch ...`.
Đổi GPU/CPU/phần mềm được ghi trong `sessions/`; vẫn dùng cùng lượt nếu cấu
hình khoa học khớp. Thay batch, hàm mục tiêu hoặc dữ liệu bị từ chối tiếp tục
chung. Nếu A100 ngắt giữa batch, chỉ làm lại phần sau checkpoint đã ghi.
Khi hết giới hạn thời gian tự đặt, ô tạm dừng; có thể xuất kết quả đang có ở ô 7.


In [ ]:
if "images70" not in globals() or "runtime_policy" not in globals():
    raise RuntimeError("Thiếu dữ liệu/cấu hình: chạy Run all từ đầu.")
from dataclasses import replace
completed_roots = {}
session_deadline = exp.time.monotonic() + MAX_SESSION_HOURS * 3600 if MAX_SESSION_HOURS > 0 else None
paused = False
for name in RUN_PROFILES:
    cfg = exp.profile_config(name)
    if cfg.split == "all70k":
        images, labels, metadata = images70, labels70, metadata70
    else:
        images, labels, metadata = exp.load_data(DATA_ROOT, CANONICAL_HASHES, cfg.split)
    print("\nĐANG CHẠY HỒ SƠ:", name, "N=", len(images), "seeds=", cfg.seeds)
    remaining_hours = max(0., (session_deadline - exp.time.monotonic()) / 3600) if session_deadline else 0.
    if session_deadline and remaining_hours <= 0:
        print("Đã hết giới hạn thời gian tự đặt. Run all lần sau để tiếp tục.")
        paused = True
        break
    current_runtime = replace(runtime_policy, resume_folder=RESUME_FOLDERS.get(name), max_session_hours=remaining_hours)
    try:
        result_root = exp.run_experiment(images, labels, metadata, cfg, OUTPUT_ROOT, runtime=current_runtime)
    except exp.PauseRequested as exc:
        print(str(exc))
        paused = True
        break
    completed_roots[name] = str(result_root)
    exp.atomic_json(OUTPUT_ROOT / "latest_completed_roots.json", completed_roots)
    from IPython.display import display
    import pandas as pd
    display(pd.read_csv(result_root / "summary.csv"))
    exp.plots(result_root)
print("Đã tạm dừng có checkpoint." if paused else "Đã hoàn thành các hồ sơ đã chọn.")
print("Có thể xuất gói kết quả đang có ở ô 7.")

## 6. Đọc kết quả và kiểm tra tác động tăng dữ liệu

CSV/JSON lưu chỉ số dạng tỷ lệ, ví dụ 0,45 = 45%. Báo cáo đủ mọi seed, gồm cả
lượt suy biến. Kiểm định chính là ARI của ADA-IM trừ IM trên `paper70k`; những
so sánh khác là bổ sung. Nếu đổi GPU/phần mềm, xem cột `mixed_environment` và
`campaign_mixed_environment`; các phép kiểm định được giữ để thăm dò nhưng
không tự gắn kết luận xác nhận cho lượt pha trộn môi trường. Khoảng tin cậy chứa 0 nghĩa là chưa có bằng chứng rõ
về chiều cải thiện với phép đo này; không kết luận chỉ từ seed tốt nhất.

Bảng 10k → 70k dưới đây dùng các lượt **chạy mới cùng bộ thực thi**, không
ghép điểm cũ vốn khác lịch ngẫu nhiên. Đây là kiểm tra quy mô dữ liệu trên các
tập có phần giao nhau, không phải đánh giá độc lập hoặc kiểm định chuyển miền.


In [ ]:
import pandas as pd
from IPython.display import display
if "completed_roots" not in globals():
    raise RuntimeError("Chưa chạy ô 5. Nếu chỉ muốn lấy kết quả cũ, chạy ô 7 sau khi gắn Drive và nạp module.")
for name, path in completed_roots.items():
    root = Path(path)
    completeness = json.loads((root / "completeness.json").read_text())
    print(name, completeness)
    assert completeness["complete"], "Thiếu lượt chạy: chưa được coi là kết quả cuối."
    pairs = pd.read_json(root / "paired_comparisons.json")
    display(pairs[["comparison", "metric", "n", "mean_delta", "ci95_low", "ci95_high", "sign_flip_p_two_sided", "planned_primary", "campaign_mixed_environment", "confirmatory"]])
if "legacy10k" in completed_roots and "legacy70k" in completed_roots:
    small = pd.read_csv(Path(completed_roots["legacy10k"]) / "per_seed.csv")
    full = pd.read_csv(Path(completed_roots["legacy70k"]) / "per_seed.csv")
    scale = small[["seed", "method", "ARI", "NMI", "ACC"]].merge(full[["seed", "method", "ARI", "NMI", "ACC"]], on=["seed", "method"], suffixes=("_10k", "_70k"), validate="one_to_one")
    for metric in ["ARI", "NMI", "ACC"]:
        scale[metric + "_delta_70k_minus_10k"] = scale[metric + "_70k"] - scale[metric + "_10k"]
    scale.to_csv(OUTPUT_ROOT / "scale_comparison_paired_seeds.csv", index=False)
    display(scale)
print("Không tự động kết luận phương pháp tốt hơn. Đọc từng seed, chênh lệch, CI, suy biến và thời gian.")

## 7. Xuất kết quả, kể cả khi chiến dịch chưa xong
Ô này có thể chạy riêng sau khi đã gắn Drive và nạp module, không yêu cầu GPU hoặc hoàn tất ô 5. Gói nhẹ có nhật ký đổi môi trường. Checkpoint/embedding vẫn giữ trên Drive; bảng thiếu seed chỉ là tạm thời.

In [ ]:
if "OUTPUT_ROOT" not in globals() or "exp" not in globals():
    raise RuntimeError("Chạy ô 1–3 để gắn Drive và nạp module trước khi xuất kết quả.")
display(pd.DataFrame(exp.progress_inventory(OUTPUT_ROOT)))
archive = exp.export_review(OUTPUT_ROOT)
print("Gói kết quả:", archive)
print("Kích thước MB:", round(archive.stat().st_size / 1024**2, 2))
print("Bạn có thể lấy ZIP từ Google Drive và gửi lại để phân tích. Không cần gửi toàn bộ checkpoint.")